# Scratch Notebook: OOV Queries on SNOMED CT \w HiT & OnT

The purpose of this notebook is to provide a walkthrough for the core experimental implementation contained within the repository.

The notebook is split into two portions. The first half contains the majority of the required source core for utilities, data management, mathematical operations, and implementations for retriever classes that enable experimental runs. One may think of this first portion as a guide to the underlying implementation. For this reason, additional intuition is also provided in Markdown between some code blocks (where deemed appropriate).

The second half of the notebook focuses on the processes of embedding and retrieval, providing the notebook cells which contain experimental results as shown in the paper. Moreover, this portion of the notebook should also serve as an example that allows readers to identify patterns that may be adapted and re-used with the existing utils that are shipped.

Note that experimental results can also be reproduced by completing `.env`, i.e., providing an NHS TRUD API key (this is optional, see the below note), then simply running `make` (as is outlined within the [README.md](../README.md)). However, **we advise that this is performed within a fresh VM or on a GPU cloud instance**, since the included deployment scripts may affect existing conda environments. Support for Docker should also be included, see the [README.md](../README.md) for more details.

<small>*Note: Including an NHS TRUD API key is optional when running `make` from the repo root. However, without an API key (which is required for licensure reasons, i.e., we cannot redistribute SNOMED CT in its entirety) the scripts will failover to a publicly available, slightly older SNOMED CT version. In this instance experimental results will then contain some small degree of variance.*</small>

### Outline

**Walkthrough Portion:**

* [Imports](#imports)
* [Utils](#basic-utils)
* [Math Functools](#math-functions)
* [Data](#data-mapping--management)
* [Retriever Classes](#retrievers)
* [Evaluation Metrics](#evaluation-metrics)

**Experimental Procedures:**

* [Embedding](#embedding--retrieval)
* [Retrieval](#retrieval--ranking)
* [Experimental Runs](#experiments)

## Imports

In [1]:
from __future__ import annotations
from typing import Union, NamedTuple, Any, Sequence, Callable, override, overload

from multiprocessing import Pool, cpu_count

from abc import ABC, abstractmethod
from collections.abc import Generator, MutableMapping, Mapping, Sequence

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import auc as sk_auc
from scipy.sparse import coo_matrix
from rank_bm25 import BM25Okapi

from sentence_transformers import SentenceTransformer

from argparse import ArgumentParser, Namespace
from pathlib import Path
from functools import reduce

from pydantic import validate_call
from tqdm import tqdm

import math
import numpy as np
import statistics

import json
import pickle
import logging
import sys
import copy
import re

import torch

from hierarchy_transformers import HierarchyTransformer
from OnT.OnT import OntologyTransformer

In [2]:
VERBOSE = False
! pip install json2html

import json2html
import latextable
from latextable import texttable

## Basic Utils

*Source: [data_utils.py](../src/hroov/utils/data_utils.py)*

Small helper functions used for tokenisation and stripping higher level concept tags (e.g. '$Hear\ Disease\ (Condition) \rightarrow Heart\ Disease$') for leakage prevention.

In [3]:
_regex_parens = re.compile(r"\s*\([^)]*\)") # for parentheses removal (prevents leakage)

def strip_parens(s: str) -> str:
    return _regex_parens.sub("", s)

def load_json(file_path: Path) -> dict[str, str]:
    with file_path.open('r', encoding='utf-8') as fp:
        return json.load(fp)

def save_json(file_path: Path, payload: dict | list, encoding: str = "utf-8", indentation: int = 4) -> None:
    with open(file_path, "w", encoding=encoding) as fp:
        json.dump(payload, fp, indent=indentation)

def load_concepts_to_list(concepts_file_path: Path) -> list[str]:
    return list(load_json(concepts_file_path).values())

def naive_tokenise(seq: str) -> list[str]:
    return seq.lower().split()

def parallel_tokenise(seq_list: list[str], workers: int, chunksize: int = 25000) -> list[list[str]]:
    with Pool(workers) as pool:
        return list(pool.map(naive_tokenise, seq_list, chunksize=chunksize))
    
def produce_candidate_ids_for_embs(embeddings_ds):
    return np.arange(len(embeddings_ds))

## Indexing Utils

*Source: [retrievers.py](../src/hroov/utils/retrievers.py)*

In [4]:
# weighted TF-IDF at run-time
def aggregate_posting_scores(query_weights, inverted):
    scores = {}
    for term, weight in query_weights.items():
        if term not in inverted:
            continue
        for doc_id, tfidf_score in inverted[term]:
            scores[doc_id] = scores.get(doc_id, 0.0) + weight * tfidf_score
    return scores

# sort TF-IDF result set
def topk(scores: dict[int, float], k: int = 10):
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)[:k]

def build_tf_idf_index(axiom_list: list[str], tfidf_dest: str, *args, **kwargs):
    vectoriser = TfidfVectorizer(**kwargs)
    doc_term_matrix = vectoriser.fit_transform(axiom_list)
    vocab = vectoriser.get_feature_names_out()
    # prep for storing to disk: create empty postings struct
    inverted_index: dict[str, list[tuple[int, float]]] = {term: [] for term in vocab}
    # see: https://matteding.github.io/2019/04/25/sparse-matrices/
    coo = coo_matrix(doc_term_matrix)
    # populate the inverted index
    for row, col, score in zip(coo.row, coo.col, coo.data):
        inverted_index[str(vocab[col])].append((int(row), float(score)))
    # order: desc
    for postings in inverted_index.values():
        postings.sort(key=lambda x: x[1], reverse=True)
    # save to disk
    with open(tfidf_dest, "wb") as fp:
        pickle.dump(
        {
            "vectorizer": vectoriser,
            "postings": postings, # type: ignore
            "verbalisations": axiom_list
        },
        fp,
        protocol=pickle.HIGHEST_PROTOCOL,
    )
    return vectoriser, inverted_index

<style>
*{
    line-height: 24px;
}
</style>

## Math Functions

*Source: [math_functools.py](../src/hroov/utils/math_functools.py)*

**L2 distance:**

$$ L_2 = \| v - u \|_2 \equiv \sqrt{\sum_{i=1}^d (v_i - u_i)^2} $$

**Inner product:**

$$\sum_{i=1}^{d} u_i \cdot v_i $$

**Cosine Similarity:**

$$ sim(u,v) = \operatorname{cos}(\theta) = \frac{u \cdot v }{\| u \|_2 \| v \|_2} $$

**Geodesic Distance (\w adaptive/sectional curvature $\kappa$):**

$$
d_{\kappa}(u,v) = \frac{1}{\sqrt{\kappa}} 
\cdot 
\operatorname{arcosh} 
\Biggl( 1 + \frac{2\kappa \|u - v\|^{2}}{
  \bigl( 1 - \kappa \|u\|^{2} \bigr) \cdot \bigl( 1 - \kappa \|v\|^{2} \bigr)
}
\Biggr),
\qquad
\|u\|,\|v\|<\frac{1}{\sqrt{\kappa}}
$$

In [5]:
def batch_euclidian_l2_distance(u: np.ndarray, vs: np.ndarray) -> np.ndarray:
    return np.linalg.norm(u - vs, axis=1)

def l2_norm(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float32)
    return np.sqrt(np.sum(x**2))

def batch_l2_norm(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float32)
    return np.asarray(np.sqrt(np.sum(x**2, axis=1)))

def inner_product(p_u: np.ndarray, p_v: np.ndarray) -> np.ndarray:
    u = np.asarray(p_u, dtype=np.float32)
    v = np.asarray(p_v, dtype=np.float32)
    return np.inner(u, v)

def batch_inner_product(p_u: np.ndarray, p_vs: np.ndarray) -> np.ndarray:
    u = np.asarray(p_u, dtype=np.float32).ravel()
    vs = np.asarray(p_vs, dtype=np.float32)
    return vs.dot(u)

def cosine_similarity(u, v, normalised=True):
    u = np.asarray(u, dtype=np.float32)
    v = np.asarray(v, dtype=np.float32)
    return np.inner(u, v) if normalised else np.inner(u, v) / (l2_norm(u) * l2_norm(v))

def batch_cosine_similarity(p_u, p_vs, normalised=True):
    u  = np.asarray(p_u,  dtype=np.float32)
    vs = np.asarray(p_vs, dtype=np.float32)
    return batch_inner_product(u, vs) if normalised else batch_inner_product(u, vs) / (l2_norm(u) * batch_l2_norm(vs))

<style>
*{
    line-height: 24px;
}
</style>

## Hyperbolic Space

A Riemannian manifold $\mathcal{M}$, of dimension $d$ is defined as a smooth (differentiable) manifold equipped with a Riemannian metric tensor $g$, such that the manifold is represented by $(\mathcal{M}, g)$. For any point $x \in \mathcal{M}$, there exists a local neighbourhood whose geometry resembles Euclidean geometry. Hyperbolic space $\mathbb{H}^n$ is a Riemannian manifold with a constant sectional curvature $-\kappa$, which can be represented in the Poincaré ball model whose points lie within the open ball, given by:
 
$$
B_{\kappa}^n = \{\ x \in \mathbb{R}^n : \|x\| < r\ \}, \qquad r=\frac{1}{\sqrt{\kappa}},
$$

where $r$ is the radius of the ball. The Poincaré metric $g_{\kappa}$ induces the hyperbolic distance function $d_{\kappa}$ between any two points $x,y \in B^n_{\kappa}$, applied for scoring in §\ref{sec:task-definition-and-methodology}, is given by:


$$
d_{\kappa}(x,y) = \frac{1}{\sqrt{\kappa}} 
\cdot 
\operatorname{arcosh} 
\Biggl( 1 + \frac{2\kappa \|x - y\|^{2}}{
  \bigl( 1 - \kappa \|x\|^{2} \bigr) \cdot \bigl( 1 - \kappa \|y\|^{2} \bigr)
}
\Biggr),
\qquad
\|x\|,\|y\|<\frac{1}{\sqrt{\kappa}}.
$$

As $\|x\|$ and $\|y\|$ approach the boundary of the ball (norm $\to \frac{1}{\sqrt{\kappa}}$), distances diverge even if the Euclidean norm difference $\|x-y\|$ is not itself significant, meaning that points situated near the boundary can represent more specific concepts (since their hyperbolic separation becomes large). This is in contrast to points situated toward the center, that represent more generic concepts.

In [6]:
def batch_poincare_distance_with_curv_k(u: np.ndarray, vs: np.ndarray, k: np.float64 | np.float32) -> np.float64 | np.float32:
    u_norm_sqd = np.sum(u**2)
    vs_norms_sqd = np.sum(vs**2, axis=1)
    l2_dist_sqd = np.sum((u - vs)**2, axis=1)
    offset = 1e-7 # tiny-offset: guard agaisnt division by zero & floating point arithmatic inaccuracies
    arg = 1 + ((2 * k * l2_dist_sqd) / ((1 - (k * u_norm_sqd + offset)) * (1 - (k * vs_norms_sqd + offset)))) # acosh
    arg = np.maximum(1.0, arg) # bounds check: domain of acosh is bound to [1, \inf)
    acosh_scaling = np.float64(1) / np.float64(np.sqrt(k)) # scaling factor: k
    return (acosh_scaling * np.arccosh(arg, dtype=np.float64)) # 1 / sqrt(k) * acosh(arg)

def batch_poincare_dist_with_adaptive_curv_k(u: np.ndarray, vs: np.ndarray, model: HierarchyTransformer | OntologyTransformer, **kwargs):
    if isinstance(model, HierarchyTransformer):
        k = np.float64(model.get_circum_poincareball(model.embed_dim).c)
    elif isinstance(model, OntologyTransformer):
        hierarchy_model = model.hit_model
        hierarchy_poincare_ball = hierarchy_model.get_circum_poincareball(hierarchy_model.embed_dim)
        k = np.float64(hierarchy_poincare_ball.c)    
    else:
        raise Exception("Hyperbolic distance should only be only calculated in B^n or H^n")
    return np.asarray(batch_poincare_distance_with_curv_k(u, vs, k))

<style>
*{
    line-height: 24px;
}
</style>

## Subsumption Score

In [7]:
def identity(x):
    return x

def subsumption_score_hit(hit_transformer: HierarchyTransformer, child_emb: np.ndarray | torch.Tensor, parent_emd: np.ndarray | torch.Tensor, centri_weight: float = 1.0):
    child_emb_t = torch.Tensor(child_emb)
    parent_emb_t = torch.Tensor(parent_emd)
    dists = hit_transformer.manifold.dist(child_emb_t, parent_emb_t)
    child_norms = hit_transformer.manifold.dist0(child_emb_t)
    parent_norms = hit_transformer.manifold.dist0(parent_emb_t)
    return -(dists + centri_weight * (parent_norms - child_norms))

def subsumption_score_ont(ontology_transformer: OntologyTransformer, child_emb: np.ndarray | torch.Tensor, parent_emb: np.ndarray | torch.Tensor, weight_lambda: float = 1.0):
    child_emb_t = torch.Tensor(child_emb)
    parent_emb_t = torch.Tensor(parent_emb)
    return ontology_transformer.score_hierarchy(child_emb_t, parent_emb_t, weight_lambda)

def entity_subsumption(u: np.ndarray, vs: np.ndarray, model: HierarchyTransformer, *, weight: float = 0.35):
    return np.asarray(subsumption_score_hit(model, u, vs, centri_weight=weight))

def concept_subsumption(u: np.ndarray, vs: np.ndarray, model: OntologyTransformer, *, weight: float = 0.35, **kwargs):
    return np.asarray(subsumption_score_ont(model, u, vs, weight_lambda=weight))

<style>
*{
    line-height: 24px;
}
</style>

## Data Mapping & Management

*Source: [query_utils.py](../src/hroov/utils/query_utils.py)*

In [8]:
def make_signature(obj):
  if isinstance(obj, Mapping):
    return tuple((key, make_signature(obj[key])) for key in sorted(obj))
  elif isinstance(obj, Sequence) and not isinstance(obj, (str, bytes, bytearray)):
    return tuple(make_signature(item) for item in obj)
  # else:
  return obj

def unique_unhashable_object_list(obj_xs: list[dict]) -> list[dict]:
    object_signatures = set()
    unique_obj_list = []
    for obj in obj_xs:
      signature = make_signature(sorted(obj.items()))
      if signature not in object_signatures:
        object_signatures.add(signature)
        unique_obj_list.append(obj)
    return unique_obj_list

def obj_max_depth(x: int, obj: Any, key: str = "depth") -> int:
  return x if x > obj[key] else obj[key]

def dcg_exp_relevancy_at_pos(relevancy: int, rank_position: int) -> float:
  if relevancy <= 0:
    return float(0.0)
  numerator = (2**relevancy) - 1
  denominator = math.log2(rank_position + 1)
  return float(numerator / denominator)

def dcg_linear_relevancy_at_pos(relevancy: int, rank_position: int) -> float:
  if relevancy <= 0:
    return float(0.0)
  numerator = relevancy
  denominator = math.log2(rank_position + 1)
  return float(numerator / denominator)

def accumulate(a, b, key='dcg'):
  return a + b[key]

#### Data Mapping: Query Models

**Query Base Model**

In [9]:
class Query:
  
  _query_obj_repr: dict
  _query_string: str
  _target: dict[str, Union[str, int]]
  _entity_mention: dict[str, Union[str, int]]

  def __init__(self, query_obj_repr: dict):
    self._query_obj_repr = query_obj_repr
    self._query_string = query_obj_repr['entity_mention']['entity_literal']
    self._target = query_obj_repr['target_entity']
    self._entity_mention = query_obj_repr['entity_mention']

  def get_query_string(self) -> str:
    return self._query_string
  
  def get_target_iri(self) -> str:
    return str(self._target['iri'])
  
  def get_target_label(self) -> str:
    return str(self._target['rdfs:label'])
  
  def get_target_depth(self) -> int:
    return int(self._target['depth'])
  
  def get_target(self) -> dict:
    return self._target

**Equivalent Query Model**

In [10]:
class EquivQuery(Query):
  
  _equiv_class_expression: Union[str, None]

  def __init__(self, query_obj_repr: dict):
    super().__init__(query_obj_repr)

**Subsumption Query Model**

In [11]:
class SubsumptionQuery(Query):
  
  _parents: list
  _ancestors: list

  def __init__(self, query_obj_repr: dict):
    super().__init__(query_obj_repr)
    self._set_parents()
    self._set_ancestors()

  def _set_parents(self):
    if self._query_obj_repr['parent_entities'] and len(self._query_obj_repr['parent_entities']) > 0:
      self._parents = self._query_obj_repr['parent_entities']
    else:
      self._parents = []

  def _set_ancestors(self):
    if self._query_obj_repr['ancestors'] and len(self._query_obj_repr['ancestors']) > 0:
      self._ancestors = self._query_obj_repr['ancestors']
    else:
      self._ancestors = []

  def get_parents(self) -> list:
    return self._parents
  
  def get_ancestors(self) -> list:
    return self._ancestors
  
  def get_all_subsumptive_targets(self) -> list:
    return [self._target, *self._parents, *self._ancestors]
  
  def get_unique_subsumptive_targets(self) -> list:
    return unique_unhashable_object_list(
      self.get_all_subsumptive_targets()
    )

  def get_sorted_subsumptive_targets(self, key="depth", reverse=False, depth_cutoff=3) -> list:
    xs = self.get_all_subsumptive_targets()
    xs.sort(key=lambda x: x[key], reverse=reverse)
    return xs[:depth_cutoff]
  
  def get_unique_sorted_subsumptive_targets(self, key="depth", reverse=False, depth_cutoff=3) -> list:
    return unique_unhashable_object_list(
      self.get_sorted_subsumptive_targets(key=key, reverse=reverse, depth_cutoff=depth_cutoff)
    )
  
  def get_targets_with_dcg(self, type="exp", depth_cutoff=3, **kwargs) -> tuple[float, list[dict]]:
    # get targets (target, parents, ancestors) ordered in ascending via depth
    targets_asc_depth = self.get_unique_sorted_subsumptive_targets(key="depth", depth_cutoff=depth_cutoff)
    # increase depth by 1 (offsetting zero-based index)
    targets_w_offset = [
      {**x, "depth": x["depth"] + 1}
      for x in targets_asc_depth
    ]
    # find the max depth (to calculate relevancy): (max_depth - depth_at_pos_k) + zero_based_offset
    # which we refer to as: relevancy := ascent height + zero_based_offset
    max_target_depth = reduce(
      obj_max_depth, 
      targets_w_offset, 
      0
    )
    # calculate relevance for each target (node/parent/ancestor) 
    targets_with_rel = [
      {**x, "relevance": (max_target_depth - x["depth"]) + 1}
      for x in targets_w_offset
    ]
    # ensure targets are sorted by relevance
    targets_with_rel.sort(key=lambda x: x['relevance'], reverse=True)
    # calculate dcg:
    if type == "linear":
      targets_with_dcg = [
        {**x, "dcg": dcg_linear_relevancy_at_pos(x['relevance'], rank)}
        for rank, x in enumerate(targets_with_rel, start=1)
      ]
    else:
      targets_with_dcg = [
        {**x, "dcg": dcg_exp_relevancy_at_pos(x['relevance'], rank)}
        for rank, x in enumerate(targets_with_rel, start=1)
      ]
    iDCG = reduce(accumulate, targets_with_dcg, 0)
    self._idcg = iDCG
    self._targets_with_dcg = targets_with_dcg
    return iDCG, targets_with_dcg

  def get_ideal_dcg(self, type="exp"):
    if self._idcg:
      return self._idcg
    iDCG, targets = self.get_targets_with_dcg()
    return iDCG

**QueryObjectMapping**

In [12]:
class QueryObjectMapping:

  _loaded: bool
  _data_file_path: Path
  _data: list
  _equiv_queries: list
  _subsumpt_queries: list

  @validate_call
  def __init__(self, json_data_fp: Path):
    self._loaded = False
    self._data_file_path = json_data_fp
    self._load()
    self._map()

  def _load(self) -> None:
    with self._data_file_path.open('r', encoding='utf-8') as fp:
      self._data = json.load(fp)
    self._loaded = True

  @validate_call
  def load_from_path(self, json_data_fp: Path) -> None:
    # overwrite an existing file path
    self._data_file_path = json_data_fp
    self._load()

  # equivalence_retrieval: bool = True, subsumption_retrieval: bool = False
  def _map(self) -> None:
    equiv_queries = []
    subsumpt_queries = []
    for query_obj_repr in self._data:
      # if the query obj within the data contains an equiv class
      if len(query_obj_repr['equivalent_classes']) > 0:
        # we're dealing with an equiv query
        equiv_queries.append(EquivQuery(query_obj_repr))
      else:
        subsumpt_queries.append(SubsumptionQuery(query_obj_repr))
    self._equiv_queries = equiv_queries
    self._subsumpt_queries = subsumpt_queries

  def get_queries(self) -> tuple[list, list]:
    return (self._equiv_queries, self._subsumpt_queries)
  
  def get_subsumpt_queries_with_no_transformations(self):
    tmp_subsumpt_queries = copy.deepcopy(self._subsumpt_queries)
    result_queries = []
    for query in tmp_subsumpt_queries:
      if query._entity_mention['transformed_entity_literal_for_type_alignment'] == "":
        result_queries.append(query)
    return result_queries
  
  def get_subsumpt_queries_with_transformations_only(self):
    tmp_subsumpt_queries = copy.deepcopy(self._subsumpt_queries)
    result_queries = []
    for query in tmp_subsumpt_queries:
      if query._entity_mention['transformed_entity_literal_for_type_alignment'] != "":
        result_queries.append(query)
    return result_queries

**QueryResult**

In [13]:
class QueryResult(NamedTuple):
  rank: int
  iri: str
  score: float
  verbalisation: str

<style>
*{
    line-height: 24px;
}
</style>

## Retrievers

*Source: [retrievers.py](../src/hroov/utils/retrievers.py), also see: [gpu_retrievers.py](../src/hroov/utils/gpu_retrievers.py) for GPU accelerated retrieval.*

**BaseRetriever**

In [14]:
class BaseRetriever(ABC):
    
  _verbalisations: list
  _meta_map: list

  @validate_call
  def __init__(self, verbalisations_fp: Path, meta_map_fp: Path):
    with open(verbalisations_fp, 'r', encoding='utf-8') as fp:
      self._verbalisations = json.load(fp)
    with open(meta_map_fp, 'r', encoding='utf-8') as fp:
       self._meta_map = json.load(fp)

  @abstractmethod
  def retrieve(self, query_string: str, *, top_k: int = 10, **kwargs) -> list[QueryResult]:
    pass

**BaseModelRetriever**

In [15]:
class BaseModelRetriever(BaseRetriever):
   
  _embeddings: np.ndarray
  _candidate_indicies: np.ndarray
  _model: Union[SentenceTransformer, HierarchyTransformer, OntologyTransformer]
  _score_fn: Callable

  @overload
  def __init__(self, verbalisations_fp: Path, meta_map_fp: Path, embeddings_fp: Path): ...

  @overload
  def __init__(self, verbalisations_fp: Path, meta_map_fp: Path, embeddings_fp: Path, *, score_fn: Callable | None = None): ...
    
  @overload
  def __init__(self, verbalisations_fp: Path, meta_map_fp: Path, embeddings_fp: Path, *, score_fn: Callable | None = None, model_fp: Path | None = None): ...
    
  @overload
  def __init__(self, verbalisations_fp: Path, meta_map_fp: Path, embeddings_fp: Path, *, score_fn: Callable | None = None, model_fp: Path | None = None, model_str: str | None = None): ...

  @validate_call
  def __init__(self, verbalisations_fp: Path, meta_map_fp: Path, embeddings_fp: Path, *, score_fn: Callable | None = None, model_fp: Path | None = None, model_str: str | None = None, **kwargs):
    super().__init__(verbalisations_fp, meta_map_fp)
    self._embeddings = np.load(embeddings_fp, mmap_mode="r")
    self._candidate_indicies = np.arange(len(self._embeddings))
    if score_fn:
      self.register_score_function(score_fn)
    if model_fp:
      try:
        self.register_local_model(model_fp.expanduser().resolve())
      except FileNotFoundError:
        self.register_model(str(model_fp), **kwargs)
    elif (not model_fp and model_str):
      self.register_model(model_str, **kwargs)

  def register_score_function(self, score_fn: Callable):
    self._score_fn = score_fn

  def get_score_function_str_or_bool(self):
    if not self._score_fn:
        return False
    # else:
    return self._score_fn.__name__

  @override
  def retrieve(self, query_string: str, *, top_k: int | None = None, reverse_candidate_scores=False, **kwargs) -> list[QueryResult]:
    """
    TODO: 1. add docstring explaining why **kwargs is accepted and pass through to _score_fn
          2. add explaination of parameters
          3. types (args/return)
    ----------------------------------------
    TODO: method is too verbose (multiple elif branches, etc), look @ providing a more concise implementation
    """
    query_embedding = self._embed(query_string)
    scored_embeddings = self._score_fn(query_embedding, self._embeddings, **kwargs)
    if reverse_candidate_scores and top_k is not None:
      top_k_indicies = self._candidate_indicies[np.flip(np.argsort(scored_embeddings))[:top_k]]
    elif not reverse_candidate_scores and top_k is not None:
      top_k_indicies = self._candidate_indicies[np.argsort(scored_embeddings)[:top_k]]
    elif reverse_candidate_scores and top_k is None:
      top_k_indicies = self._candidate_indicies[np.flip(np.argsort(scored_embeddings))]
    elif not reverse_candidate_scores and top_k is None:
      top_k_indicies = self._candidate_indicies[np.argsort(scored_embeddings)]
    else:
      raise KeyError("Valid arguments for reverse_candidate_scores and top_k must be set.")
    results = []
    for rank, candidate_index in enumerate(top_k_indicies):
      candidate_score = scored_embeddings[candidate_index]
      candidate_meta_map = self._meta_map[candidate_index]
      candidate_verbalisation = candidate_meta_map['verbalisation']
      candidate_iri = candidate_meta_map['iri']
      results.append((rank, candidate_iri, candidate_score, candidate_verbalisation))
    return results

  @abstractmethod
  def register_model(self, model: str) -> None: ...

  @abstractmethod
  @overload
  def register_model(self, model: str, **kwargs) -> None:
    pass

  @abstractmethod
  def register_local_model(self, model_fp: Path) -> None:
    pass

  @abstractmethod
  def _embed(self, query_string: str) -> np.ndarray:
    pass

**HiTRetriever**

In [16]:
class HiTRetriever(BaseModelRetriever):

  @override
  def register_model(self, model: str) -> None:
    self._model = HierarchyTransformer.from_pretrained(model)

  @override
  def register_local_model(self, model_fp: Path) -> None:
    self._model = HierarchyTransformer.from_pretrained(str(model_fp.expanduser().resolve()))  

  @override
  def _embed(self, query_string: str) -> np.ndarray:
    return (self._model.encode(
      [query_string]
    ).astype("float32"))[0] # type: ignore

**OnTRetriever**

In [17]:
class OnTRetriever(BaseModelRetriever):

  @override
  def register_model(self, model: str, **kwargs) -> None:
    self._model = OntologyTransformer.from_pretrained(model, **kwargs)

  @override
  def register_local_model(self, model_fp: Path) -> None:
    self._model = OntologyTransformer.from_pretrained(str(model_fp.expanduser().resolve()))

  @override
  def _embed(self, query_string: str) -> np.ndarray:
    return (self._model.encode_concept(
      [query_string]
    ).astype("float32"))[0] # type: ignore

**SBERTRetriever**

In [18]:
class SBERTRetriever(BaseModelRetriever):

  @override
  def register_model(self, model: str) -> None:
    self._model = SentenceTransformer.load(model)

  @override
  def register_local_model(self, model_fp: Path) -> None:
    self._model = SentenceTransformer.load(str(model_fp.expanduser().resolve()))

  @override
  def _embed(self, query_string: str) -> np.ndarray:
    return (self._model.encode(
      [query_string]
    ).astype("float32"))[0] # type: ignore

In [19]:
from transformers import AutoTokenizer, AutoModel
import torch

class SapBERTRetriever(BaseModelRetriever):
    
    _tokenizer: AutoTokenizer
    _model: AutoModel
    _device: torch.device

    @override
    def register_model(self, model: str, **kwargs) -> None:
        self._device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self._tokenizer = AutoTokenizer.from_pretrained(model)
        self._model = AutoModel.from_pretrained(model, **kwargs).to(self._device)
        self._model.eval()

    @override
    def register_local_model(self, model_fp: Path) -> None:
        resolved_path = str(model_fp.expanduser().resolve())
        self._device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self._tokenizer = AutoTokenizer.from_pretrained(resolved_path)
        self._model = AutoModel.from_pretrained(resolved_path).to(self._device)
        self._model.eval()

    @override
    def _embed(self, query_string: str) -> np.ndarray:
        inputs = self._tokenizer(
            query_string,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt"
        ).to(self._device)
        
        with torch.no_grad():
            outputs = self._model(**inputs)
        
        # [CLS] token embedding, normalised
        cls_embedding = outputs.last_hidden_state[:, 0, :]
        cls_embedding = torch.nn.functional.normalize(cls_embedding, p=2, dim=-1)
        
        return cls_embedding.squeeze(0).cpu().numpy().astype("float32")

**BM25Retriever**

In [20]:
class BM25Retriever(BaseRetriever):
    
  _bm25: BM25Okapi

  @validate_call
  def __init__(self, verbalisations_fp: Path, meta_map_fp: Path, k1: float = 1.3, b: float = 0.7):
    super().__init__(verbalisations_fp, meta_map_fp)
    self._tokenised_verbalisations = parallel_tokenise(self._verbalisations, workers=4)
    self._bm25 = BM25Okapi(self._tokenised_verbalisations, k1=k1, b=b)

  @classmethod
  def build_from_index(cls, index_fp: Path | str):
    pass

  def save_index(self, index_fp: Path | str):
    if isinstance(index_fp, Path):
      index_fp = str(index_fp.expanduser().resolve())
    with open(index_fp, "wb") as fp:
      pickle.dump({
        "index": self._bm25,
        "verbalisations": self._verbalisations,
        "meta_map": self._meta_map
      }, fp, protocol=pickle.HIGHEST_PROTOCOL)
    print("Saved BM25 index to disk.")
    
  def load_index(self, index_fp: Path | str):
    if isinstance(index_fp, Path):
      index_fp = str(index_fp.expanduser().resolve())
    with open(index_fp, "rb") as fp:
      bm25_bin = pickle.load(fp)
    self._bm25 = bm25_bin['index']
    self._verbalisations = bm25_bin['verbalisations']
    self._meta_map = bm25_bin['meta_map']

  def retrieve(self, query_string: str, *, top_k: int | None = None, **kwargs) -> list[QueryResult]:
    tokens = naive_tokenise(query_string)
    scores = self._bm25.get_scores(tokens)
    if top_k is not None:
      top_idx = np.argsort(scores)[::-1][:top_k]
    else:
      top_idx = np.argsort(scores)[::-1]
    results = []
    for rank, idx in enumerate(top_idx):
      iri = self._meta_map[idx]['iri']
      verbalisation = self._verbalisations[idx]
      results.append(
        QueryResult(
          rank=rank,
          iri=iri,
          score=float(scores[idx]),
          verbalisation=verbalisation
        )
      )
    return results

In [21]:
class TFIDFRetriever(BaseRetriever):
  
  # TODO: clean up and implement `save` and `load` methods, similar to BM25.

  _vectorizer: TfidfVectorizer
  _inverted_index: dict[str, list[tuple[int, float]]]
  _tfidf_matrix: "scipy.sparse.csr_matrix"
  _tokenizer: Callable[[str], Sequence[str]] | None

  @validate_call
  def __init__(self, verbalisations_fp: Path, meta_map_fp: Path, *,
    lowercase: bool = True, stop_words: str | None = "english",
    ngram_range: tuple[int, int] = (1, 1),
    tokenizer: Callable[[str], Sequence[str]] | None = None,
    max_features: int | None = None,
  ) -> None:
    super().__init__(verbalisations_fp, meta_map_fp)
    self._vectorizer = TfidfVectorizer(
      stop_words="english",
      use_idf=True,
      smooth_idf=True,
      # norm="l2"
      norm=None
    )
    doc_term_matrix = self._vectorizer.fit_transform(self._verbalisations)
    vocab = self._vectorizer.get_feature_names_out()
    inverted_index: dict[str, list[tuple[int, float]]] = {term: [] for term in vocab}
    coo = coo_matrix(doc_term_matrix)

    for row, col, score in zip(coo.row, coo.col, coo.data):
      inverted_index[str(vocab[col])].append((int(row), float(score)))
    for postings in inverted_index.values():
      postings.sort(key=lambda x: x[1], reverse=True)

    self._inverted_index = inverted_index

  def retrieve(self, query_string: str, *, top_k: int | None = None, **kwargs) -> list[QueryResult]:
    query_vec = self._vectorizer.transform([query_string])
    vocab = self._vectorizer.get_feature_names_out()        
    q_weights = {
      vocab[col]: float(val)
      for col, val in zip(query_vec.indices, query_vec.data) # type: ignore
      if val > 0.0
    }
    tfidf_scores = aggregate_posting_scores(q_weights, self._inverted_index)
    if top_k:
      tfidf_top = topk(tfidf_scores, top_k)
    else:
      tfidf_top = topk(tfidf_scores, len(tfidf_scores))
    results: list[QueryResult] = []
    for rank, (doc_id, score) in enumerate(tfidf_top, 1):
      iri = self._meta_map[doc_id]['iri']
      verbalisation = self._meta_map[doc_id]['verbalisation']
      results.append(
        QueryResult(
          rank=rank,
          iri=iri,
          score=float(score),
          verbalisation=verbalisation,
        )
      )
    return results

<style>
*{
    line-height: 24px;
}
</style>

## Evaluation Metrics

Most evaluation metrics are actually defined within the context of the experiments themselves; i.e. they're not pre-defined as functions prior to implementation. However, some intuition for nDCG is provided below.

Other evaluation metrics include:

Hit rate / H@k, MRR, MR, Median Rank, PR-AUC, mAP and Recall@k.

In [22]:
# nDCG NOTES:

# We don't want to directly modify the depth (as that might be confusing)
# So.. we're going to create a new set of subsumptive targets with a relevancy score
# where the relevancy := ascent_height + 1 \forall t \in T
# where the ascent_height := max(depth) - depth_at_t
# -> relevancy := (max(depth) - depth_at_t) + 1 \forall t \in T
#
# Example (POLYHIERARCHICAL ONTOLOGY):
#
# Say, you've got a query_string with a target entity, two parent entities and seven ancestors 
# (exclusive of SNOMED CT CONCEPT & owl:Thing):
# 
#                              ---------------------------------------------------------------------------------
#                  ENTITY_TYPE | DEPTH | ASCENT_HEIGHT | ASCENT_HEIGHT + 1 (REL) | 2^{r} - 1 |  = val  |  DCG  |
#                              |-------|---------------|-------------------------|-----------|---------|-------|
#      X      <- TARGET_ENTITY |   0   |       4       |            5            |  2^5 - 1  |    31   |  31.0 |  
#     / \                      |       |               |                         |           |         |       |
#    X   X          <- PARENTS |   1   |       3       |            4            |  2^4 - 1  |    15   |  9.46 |
#   /   / \                    |       |               |                         |           |         |       |
#  X   X   X      <- ANCESTORS |   2   |       2       |            3            |  2^3 - 1  |    7    |  7.50 |
#  |   |    \                  |       |               |                         |           |         |       |
#  |   X     X    <- ANCESTORS |   3   |       1       |            2            |  2^2 - 1  |    3    |  3.57 |
#  |   |    / \                |       |               |                         |           |         |       |
#  |   |   X   X  <- ANCESTORS |   4   |       0       |            1            |  2^1 - 1  |    1    |  1.61 |
#  |   |   |   |               ---------------------------------------------------------------------------------
#  -------------------------                     
#  | TOP SNOMED CT CONCEPT |
#  -------------------------
#  |       owl:Thing       |
#  -------------------------------------------------------------------------------------------------------
#  |              IDEAL ORDERING                   |                  EXAMPLE RESULTS                    |
#  -------------------------------------------------------------------------------------------------------
#  |  RANK      ENTITY NAME          REL    DCG    |  RANK      ENTITY               REL     DCG         |
#  -------------------------------------------------------------------------------------------------------
#  |  1         TARGET_ENTITY        5      31.0   |  1         ANCESTOR_@_3_1|2     2       3.00        |
#  |  2         PARENT_1|2           4      9.46   |  2         ANCESTOR_@_2_1|2|3   3       4.42        |
#  |  3         PARENT_2|2           4      7.50   |  3         TARGET_ENTITY        5       15.5        |
#  |  4         ANCESTOR_@_2_1|2|3   3      3.01   |  4         PARENT_1|2           4       6.46        |
#  |  5         ANCESTOR_@_2_1|2|3   3      2.71   |  5         ANCESTOR_@_2_1|2|3   3       2.71        |
#  |  6         ANCESTOR_@_2_1|2|3   3      2.49   |  6         ANCESTOR_@_2_1|2|3   3       2.49        |
#  |  7         ANCESTOR_@_3_1|2     2      1.00   |  7         ANCESTOR_@_3_1|2     2       1.00        |
#  |  8         ANCESTOR_@_3_1|2     2      0.95   |  8         NOT-RELEVANT-RESULT  0       0.00        |
#  |  9         ANCESTOR_@_4_1|2     1      0.30   |  9         ANCESTOR_@_4_1|2     1       0.30        |
#  |  10        ANCESTOR_@_4_1|2     1      0.29   |  10        NOT-RELEVANT-RESULT  0       0.00        |
# --------------------------------------------------------------------------------------------------------
#  |     iDCG = \sum_i^{\|T\|}t_{dcg} = 58.72      |  DCG@10 = \sum_i^{\|Q_{results}\|}q_dcg = 35.88     |
# --------------------------------------------------------------------------------------------------------
#
#     nDCG@10 =  DCG@10        35.88
#               --------   =   -----   =  0.611
#                iDCG@10       58.72
#
#
#  * depth is measured from the target.
#  * Quick Recap: we're taking an OOV phrase/string (from a set of entity mentions on a QA dataset)
#    -> assigning a target SNOMED entity to that entity mention, s.t. SNOMED entity ~= entity mention
#    -> for the entity mention, we traverse the ontology from the target, up the hierarchy, until we reach the top
#    -> as we traverse the structure, we record the depth, rdfs:label, pref:label alt:labels and IRI of each node
#    -> that allows us to construct a graph (that kind of looks like a tree, as its a fragment of an ontology, which is
#       largely a taxonomy, but it is polyhierarchical, so it ends up being a graph)
#    -> as we get further away from the target, the concepts get more general/broad, so assume/consider the relevancy
#       decreases monotonically as a function of the depth (we implement two relevancy scores for DCG:
#           
#           (1) Relevancy \w exponential decay:  \frac{2^{rel} - 1}{log_2(rank+1)}
#           (2) Relevancy \w linear decay: \frac{rel}{ln(rank+1)}
#
#    -> We opt to use exponential decay, as it more suitably approximates distance in hyperbolic space, though, it is noted
#       that the result is normalised anyway...

In [23]:
from functools import reduce
from typing import Any
import math

def add(a, b, key='dcg'):
  return a + b[key]

def obj_max_depth(x: int, obj: Any, key: str = "depth") -> int:
  return x if x > obj[key] else obj[key]

def dcg_exp_relevancy_at_pos(relevancy: int, rank_position: int) -> float:
  if relevancy <= 0:
    return float(0.0)
  numerator = (2**relevancy) - 1
  denominator = math.log2(rank_position + 1)
  return float(numerator / denominator)

def compute_ndcg_at_k(results: list[tuple[int, str, float, str]], targets_with_dcg_exp: list[dict], k: int = 20) -> float:
  relevance_map = {target['iri']: target['relevance'] for target in targets_with_dcg_exp}
  dcg = 0.0
  for rank, (idx, iri, score, label) in enumerate(results[:k], start=1):
    rel = relevance_map.get(iri, 0)
    dcg += dcg_exp_relevancy_at_pos(rel, rank)
  ideal_dcg = sum(target['dcg'] for target in targets_with_dcg_exp[:k])
  if ideal_dcg == 0:
    return 0.0
  
  return dcg / ideal_dcg

<style>
*{
    line-height: 24px;
}
</style>

## Embedding & Retrieval 

#### Precompute Embeddings

1. Process the entity lexicon (or $\mathcal{EL}$-normalised concept strings) and extract a verbalisation list.
2. Encode each concepts' textual representation into each models native embedding space.
3. Save the embeddings to disk, along with a map between $emb \leftrightarrow concept_{text\ repr}$.

#### Retrieval & Ranking

1. Load the embeddings + their mappings.
2. Accept a query string $q$.
3. Compute a score for $q$ using a retrieval method: $\{ TFIDF, BM25, SBERT, HiT, OnT \}$ using the pre-computed embeddings.
4. `argsort` the embs according to their scores
5. Return the sorted list.

**Embedding:**

In [24]:
# date function

from datetime import datetime

def get_date_string() -> str:
    now = datetime.now()
    return now.strftime("%d_%m_%y_%H%M")

In [25]:
# prep for embeding:

dataset_name = 'OET_evaluation_dataset_V2026_01_DISEASE_ALL.json'

print(f"Preparing data for indexing/encoding for {dataset_name}")

data_dir = "../data"

custom_embeddings_dir = f"../embeddings_4_EPOCHS_MINI_{dataset_name}_{get_date_string()}"
# custom_embeddings_dir = '../embeddings_OET_evaluation_dataset_V2026_01_CPP_ALL.json_06_03_26_1345'

if not (Path(data_dir).exists()):
  print("[WARNING] No data directory exists. The notebook will fail. Review the README.md, or the docs dir.")

path_preprocessed_entity_lexicon_US_MINI = '../data/preprocessed_entity_lexicon_US_20140901_MINI_CPP.json'
# path_preprocessed_entity_lexicon_US_FULL = '../data/preprocessed_entity_lexicon_US_20140901.json'
# path_preprocessed_entity_lexicon_INTERNATIONAL_MINI = '../data/preprocessed_entity_lexicon_INTERNATIONAL_20250901_MINI.json'
# path_preprocessed_entity_lexicon_INTERNATIONAL_FULL = '../data/preprocessed_entity_lexicon_INTERNATIONAL_20250901_FULL.json'

# if an embeddings dir has not yet been created, create one
Path(custom_embeddings_dir).expanduser().resolve().mkdir(parents=True, exist_ok=True)

# custom_source_CPP_data_dir_NIL = Path('../../HOPE/data/OET_data/MM-S14-CPP/mention-edge-pair-level/entity_lexicon_MM-S14-CPP__NIL_only.json').expanduser().resolve()
# custom_source_CPP_data_dir_TRAIN = Path('../../HOPE/data/OET_data/MM-S14-CPP/mention-edge-pair-level/entity_lexicon_MM-S14-CPP__TRAIN_CONCEPTS.json').expanduser().resolve()
# custom_source_CPP_data_dir_ALL_EDGE_CONCEPTS = Path('../../HOPE/data/CPP_entity_lexicon_no_ex_concepts.json').expanduser().resolve()

# custom_source_Disease_data_dir_NIL = Path('../../HOPE/data/OET_data/MM-S14-Disease/mention-edge-pair-level/entity_lexicon_MM-S14-Disease__NIL_only.json').expanduser().resolve()
# custom_source_Disease_data_dir_TRAIN = Path('../../HOPE/data/OET_data/MM-S14-Disease/mention-edge-pair-level/entity_lexicon_MM-S14-Disease__TRAIN_CONCEPTS.json').expanduser().resolve()
# custom_source_Disease_data_dir_ALL_EDGE_CONCEPTS = Path('../../HOPE/data/Disease_entity_lexicon_no_ex_concepts.json').expanduser().resolve()

# generated during SNOMED CT processing
# entity_lexicon_fp = Path(f"{data_dir}/preprocessed_entity_lexicon.json")

# totally custom (drawn from SNOMED 2014) entity lexicon

# custom_common_entity_lexicon = Path('../data/custom_entity_lexicon.json')

# PREVIOUS DATA

# previous_entity_lexicon = Path('../data/prev_eval_preprocessed_entity_lexicon.json')

#############################

# USE ENTITY LEXICON FOR CPP:

# entity_lexicon_fp = custom_source_CPP_data_dir_ALL_EDGE_CONCEPTS

# USE ENTITY LEXICON FOR DISEASE

# entity_lexicon_fp = custom_source_Disease_data_dir_ALL_EDGE_CONCEPTS

# USE CUSTOM ENTITY LEXICON

# entity_lexicon_fp = custom_common_entity_lexicon

#############################

# USE ENTITY LEXICON FOR OET DATA (MINI)

entity_lexicon_fp = Path(path_preprocessed_entity_lexicon_US_MINI)

# USE ENTITY LEXICON FOR OET DATA (FULL)

# entity_lexicon_fp = Path(path_preprocessed_entity_lexicon_US_FULL)

# USE ENTITY LEXICON FOR EVAL-100 DATA (MINI)

# entity_lexicon_fp = path_preprocessed_entity_lexicon_INTERNATIONAL_MINI

# USE ENTITY LEXICON FOR EVAL-100 DATA (FULL)

# entity_lexicon_fp = Path(path_preprocessed_entity_lexicon_INTERNATIONAL_FULL)

#############################

# list of the verbalisations (label text, or deeponto verbs)
verbalisation_list_fp = Path(f"{custom_embeddings_dir}/verbalisations.json")
# each index of the entity_map points to a tuple: (index, label, verbalisation, iri)
entity_map_fp = Path(f"{custom_embeddings_dir}/entity_map.json")
# compiles a list of the above mappings (handy when it comes to argsort)
entity_mappings_list_fp = Path(f"{custom_embeddings_dir}/entity_mappings.json")

entity_lexicon = load_json(entity_lexicon_fp)
iris = entity_lexicon.keys()

entity_map = {}
entity_verbalisation_list = []
list_of_entity_mappings = []

for entity_idx, entity_iri in enumerate(tqdm(iris)):
    entity_map[str(entity_idx)] = {
        "mapping_id": str(entity_idx),
        "label": entity_lexicon[entity_iri].get('name'), # type: ignore
        "verbalisation": strip_parens(str(entity_lexicon[entity_iri].get('name'))).lower(), # type: ignore
        "iri": entity_iri
    }
    entity_verbalisation_list.append(strip_parens(str(entity_lexicon[entity_iri].get('name'))).lower()) # type: ignore
    list_of_entity_mappings.append(entity_map[str(entity_idx)])

save_json(verbalisation_list_fp, entity_verbalisation_list)
save_json(entity_map_fp, entity_map)
save_json(entity_mappings_list_fp, list_of_entity_mappings)

print("Complete!")

Preparing data for indexing/encoding for OET_evaluation_dataset_V2026_01_DISEASE_ALL.json


100%|██████████| 173178/173178 [00:00<00:00, 459816.27it/s]


Complete!


In [26]:
embs_already_exist = False

In [27]:
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm
import torch
import numpy as np

sapbert_hf_string = "cambridgeltl/SapBERT-from-PubMedBERT-fulltext"

if not embs_already_exist:
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tokenizer = AutoTokenizer.from_pretrained(sapbert_hf_string)
    model = AutoModel.from_pretrained(sapbert_hf_string).to(device)
    model.eval()

    batch_size = 128
    all_embeddings = []

    for i in tqdm(range(0, len(entity_verbalisation_list), batch_size), desc="Encoding"):
        batch = entity_verbalisation_list[i : i + batch_size]
        
        inputs = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            outputs = model(**inputs)
        
        # [CLS] embeddings
        cls_embeddings = outputs.last_hidden_state[:, 0, :]
        cls_embeddings = torch.nn.functional.normalize(cls_embeddings, p=2, dim=-1)
        all_embeddings.append(cls_embeddings.cpu())

    sapbert_embeddings = torch.cat(all_embeddings, dim=0).numpy().astype("float32")
    np.save(f"{custom_embeddings_dir}/sapbert-embeddings.npy", sapbert_embeddings)

Encoding: 100%|██████████| 1353/1353 [00:27<00:00, 49.41it/s]


In [28]:
sbert_plm_hf_string = "all-MiniLM-L12-v2"

if not embs_already_exist:

  sbert_plm_encoder = SentenceTransformer.load(sbert_plm_hf_string)
  sbert_plm_embeddings = sbert_plm_encoder.encode(
      entity_verbalisation_list,
      batch_size=128,
      show_progress_bar=True,
      normalize_embeddings=True
  ).astype("float32")
  np.save(f"{custom_embeddings_dir}/sbert-plm-embeddings.npy", sbert_plm_embeddings)

Batches:   0%|          | 0/1353 [00:00<?, ?it/s]

In [29]:
hit_model_20140901_mini_fp = '/mnt/data/HOPE/models/4_EPOCH_MODELS/HiT-all-MiniLM-L12-v2-US-20140901-MINI-CPP-4-EPOCHS/final'

if not embs_already_exist:

  hit_snomed_14_mini_encoder = HierarchyTransformer.from_pretrained(hit_model_20140901_mini_fp)
  hit_snomed_14_mini_embeddings = hit_snomed_14_mini_encoder.encode(
      entity_verbalisation_list,
      batch_size=128,
      show_progress_bar=True
  ).astype("float32")
  np.save(f"{custom_embeddings_dir}/hit-snomed-14-mini-embs.npy", hit_snomed_14_mini_embeddings)

Batches:   0%|          | 0/1353 [00:00<?, ?it/s]

In [30]:
hit_model_20140901_full_fp = '/mnt/data/HOPE/models/4_EPOCH_MODELS/HiT-all-MiniLM-L12-v2-US-20140901-FULL-4-EPOCHS/final'

if not embs_already_exist:

  hit_snomed_14_full_encoder = HierarchyTransformer.from_pretrained(hit_model_20140901_full_fp)
  hit_snomed_14_full_embeddings = hit_snomed_14_full_encoder.encode(
      entity_verbalisation_list,
      batch_size=128,
      show_progress_bar=True
  ).astype("float32")
  np.save(f"{custom_embeddings_dir}/hit-snomed-14-full-embs.npy", hit_snomed_14_full_embeddings)

Batches:   0%|          | 0/1353 [00:00<?, ?it/s]

In [31]:
ont_model_20140901_mini_fp = '/mnt/data/HOPE/models/4_EPOCH_MODELS/OnTr-all-MiniLM-L12-v2-OnT-SNO-14-MINI-4-EPOCHS/final'

if not embs_already_exist:

  ont_snomed_14_mini_encoder = HierarchyTransformer.from_pretrained(ont_model_20140901_mini_fp)
  ont_snomed_14_mini_embeddings = ont_snomed_14_mini_encoder.encode(
      entity_verbalisation_list,
      batch_size=128,
      show_progress_bar=True
  ).astype("float32")
  np.save(f"{custom_embeddings_dir}/ont-snomed-14-mini-embs.npy", ont_snomed_14_mini_embeddings)

Batches:   0%|          | 0/1353 [00:00<?, ?it/s]

In [32]:
ont_model_20140901_full_fp = '/mnt/data/HOPE/models/4_EPOCH_MODELS/OnTr-all-MiniLM-L12-v2-OnT-SNO-14-FULL-4-EPOCHS/final'

if not embs_already_exist:

  ont_snomed_14_full_encoder = HierarchyTransformer.from_pretrained(ont_model_20140901_full_fp)
  ont_snomed_14_full_embeddings = ont_snomed_14_full_encoder.encode(
      entity_verbalisation_list,
      batch_size=128,
      show_progress_bar=True
  ).astype("float32")
  np.save(f"{custom_embeddings_dir}/ont-snomed-14-full-embs.npy", ont_snomed_14_full_embeddings)

Batches:   0%|          | 0/1353 [00:00<?, ?it/s]

**Retrieval**

*Load the pre-computed embeddings.*

In [33]:
sbert_embs = np.load(f"{custom_embeddings_dir}/sbert-plm-embeddings.npy", mmap_mode="r")
sapbert_embs = np.load(f"{custom_embeddings_dir}/sapbert-embeddings.npy", mmap_mode="r")

In [34]:
hit_snomed_14_mini_embs = np.load(f"{custom_embeddings_dir}/hit-snomed-14-mini-embs.npy", mmap_mode="r")
hit_snomed_14_embs = np.load(f"{custom_embeddings_dir}/hit-snomed-14-full-embs.npy", mmap_mode="r")
ont_snomed_14_mini_embs = np.load(f"{custom_embeddings_dir}/ont-snomed-14-mini-embs.npy", mmap_mode="r")
ont_snomed_14_embs = np.load(f"{custom_embeddings_dir}/ont-snomed-14-full-embs.npy", mmap_mode="r")

In [35]:
common_map = Path(f"{custom_embeddings_dir}/entity_mappings.json")
common_verbalisations = Path(f"{custom_embeddings_dir}/verbalisations.json")

**Retrievers: Lexical Methods**

In [36]:
tfidf_ret = TFIDFRetriever(common_verbalisations, common_map)
bm25_ret = BM25Retriever(common_verbalisations, common_map, k1=1.3, b=0.7)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

**SBERT Retrievers**

In [37]:
sbert_plm_hf_string = "all-MiniLM-L12-v2"

sbert_ret_plm_w_cosine_sim = SBERTRetriever(
  embeddings_fp=Path(f"{custom_embeddings_dir}/sbert-plm-embeddings.npy"),
  meta_map_fp=common_map,
  verbalisations_fp=common_verbalisations,
  model_str="all-MiniLM-L12-v2",
  score_fn=batch_cosine_similarity
)

#### SapBERT Retriever

In [38]:
sapbert_hf_string = "cambridgeltl/SapBERT-from-PubMedBERT-fulltext"

sapbert_retriever = SapBERTRetriever(
    embeddings_fp=Path(f"{custom_embeddings_dir}/sapbert-embeddings.npy"),
    meta_map_fp=common_map,
    verbalisations_fp=common_verbalisations,
    model_str=sapbert_hf_string,
    score_fn=batch_cosine_similarity
)

**HiT Retrievers**

In [39]:
hit_model_20140901_mini_fp = Path(hit_model_20140901_mini_fp)

hit_ret_snomed_14_mini_w_hyp_dist = HiTRetriever(
  embeddings_fp=Path(f"{custom_embeddings_dir}/hit-snomed-14-mini-embs.npy"),
  meta_map_fp=common_map,
  verbalisations_fp=common_verbalisations,
  model_fp=hit_model_20140901_mini_fp,
  score_fn=batch_poincare_dist_with_adaptive_curv_k
)

hit_ret_snomed_14_mini_w_ent_sub = HiTRetriever(
  embeddings_fp=Path(f"{custom_embeddings_dir}/hit-snomed-14-mini-embs.npy"),
  meta_map_fp=common_map,
  verbalisations_fp=common_verbalisations,
  model_fp=hit_model_20140901_mini_fp,
  score_fn=entity_subsumption
)

In [40]:
hit_model_20140901_full_fp = Path(hit_model_20140901_full_fp)

hit_ret_snomed_14_w_hyp_dist = HiTRetriever(
  embeddings_fp=Path(f"{custom_embeddings_dir}/hit-snomed-14-full-embs.npy"),
  meta_map_fp=common_map,
  verbalisations_fp=common_verbalisations,
  model_fp=hit_model_20140901_full_fp,
  score_fn=batch_poincare_dist_with_adaptive_curv_k
)

hit_ret_snomed_14_w_ent_sub = HiTRetriever(
  embeddings_fp=Path(f"{custom_embeddings_dir}/hit-snomed-14-full-embs.npy"),
  meta_map_fp=common_map,
  verbalisations_fp=common_verbalisations,
  model_fp=hit_model_20140901_full_fp,
  score_fn=entity_subsumption
)

In [41]:
ont_model_20140901_mini_fp = Path(ont_model_20140901_mini_fp)

ont_snomed_14_mini_w_hyp_dist = OnTRetriever(
    embeddings_fp=Path(f"{custom_embeddings_dir}/ont-snomed-14-mini-embs.npy"),
    meta_map_fp=common_map,
    verbalisations_fp=common_verbalisations,
    model_fp=ont_model_20140901_mini_fp,
    score_fn=batch_poincare_dist_with_adaptive_curv_k
)

ont_snomed_14_mini_w_con_sub = OnTRetriever(
    embeddings_fp=Path(f"{custom_embeddings_dir}/ont-snomed-14-mini-embs.npy"),
    meta_map_fp=common_map,
    verbalisations_fp=common_verbalisations,
    model_fp=ont_model_20140901_mini_fp,
    score_fn=concept_subsumption
)

In [42]:
ont_model_20140901_full_fp = Path(ont_model_20140901_full_fp)

ont_snomed_14_w_hyp_dist = OnTRetriever(
    embeddings_fp=Path(f"{custom_embeddings_dir}/ont-snomed-14-full-embs.npy"),
    meta_map_fp=common_map,
    verbalisations_fp=common_verbalisations,
    model_fp=ont_model_20140901_full_fp,
    score_fn=batch_poincare_dist_with_adaptive_curv_k
)

ont_snomed_14_w_con_sub = OnTRetriever(
    embeddings_fp=Path(f"{custom_embeddings_dir}/ont-snomed-14-full-embs.npy"),
    meta_map_fp=common_map,
    verbalisations_fp=common_verbalisations,
    model_fp=ont_model_20140901_full_fp,
    score_fn=concept_subsumption
)

# Experiments

### Configuration & Data

In [43]:
# load query objects via QueryObjectMapping:

# data_query_mapping = QueryObjectMapping(Path("../data/eval_dataset_50.json"))

# OET_evaluation_dataset_NIL_V4__THU_18_DEC_CPP_ALL.json

# OET_evaluation_dataset_NIL_V4__THU_18_DEC_DISEAE_ALL.json

# OET_evaluation_dataset_NIL_V5__THU_18_DEC_CPP_SINGLE.json

# OET_evaluation_dataset_NIL_V5__THU_18_DEC_DISEASE_SINGLE.json

#### UPDATE 2026 ####

# OET_CPP_ALL

# OET_CPP_ALL_DATASET_PATH = '../data/OET_evaluation_dataset_V2026_01_CPP_ALL.json'

# OET_DISEASE_ALL

OET_DISEASE_ALL_DATASET_PATH = '../data/OET_evaluation_dataset_V2026_01_DISEASE_ALL.json'

#####################

# EVAL_100_DATASET = '../data/clean_EVAL_100.json'

#####################

data_query_mapping = QueryObjectMapping(Path(OET_DISEASE_ALL_DATASET_PATH))

equiv_queries, subsumpt_queries = data_query_mapping.get_queries()

global_cutoff_depth = 1 # global cutoff parameter, i.e. d={1,3,5}

logs_path = f'../logs_OET_DISEASE_4_EPOCHS_MINI_{get_date_string()}'

In [44]:
models_dict = {
  # Lexical Baselines
  "BoW Lexical TFIDF": tfidf_ret,
  "BoW Lexical BM25": bm25_ret,
  # Embedding-based (Contextualised) Baseline
  "SBERT PLM cos-sim": sbert_ret_plm_w_cosine_sim,
  "SapBERT FT cos-sim": sapbert_retriever,
  # HiT Model Mini
  "HiT-MxR SNO-14-M d_k": hit_ret_snomed_14_mini_w_hyp_dist,
  "HiT-MxR SNO-14-M s_e": hit_ret_snomed_14_mini_w_ent_sub,
  # HiT Model
  "HiT-MxR SNO-14-F d_k": hit_ret_snomed_14_w_hyp_dist,
  "HiT-MxR SNO-14-F s_e": hit_ret_snomed_14_w_ent_sub,
  # OnT Model Mini
  "OnT-Prd SNO-14-M d_k": ont_snomed_14_mini_w_hyp_dist,
  "OnT-Prd SNO-14-M s_c": ont_snomed_14_mini_w_con_sub,
  # OnT Model
  "OnT-Prd SNO-14-F d_k": ont_snomed_14_w_hyp_dist,
  "OnT-Prd SNO-14-F s_c": ont_snomed_14_w_con_sub,
}

In [45]:
# PREP TABLE START #
experiment_table = texttable.Texttable()
experiment_table.set_deco(texttable.Texttable.HEADER)
experiment_table.set_precision(2)
experiment_table.set_cols_dtype(['t', 't', 't', 'f', 'f', 'f', 'f', 'f', 'f', 'f', 'f', 'f'])
experiment_table.set_cols_align(["l", "l", "l", "c", "c", "c", "c", "c", "c", "c", "c", "c"])
experiment_table.header(["Model", "Variant", "Metric", "mAP", "MRR", "H@1", "H@3", "H@5", "Med", "MR", "nDCG@10", "R@100"])
# END-PREP TABLE #

ks      = [1, 3, 5, 10, 100, len(entity_verbalisation_list)]
MAX_K   = max(ks)

all_results = {}

centripetal_weight = 0.7

for model_name, model in models_dict.items():
    
    # init accumulators
    results = {
      "MRR": 0.0, # Mean Reciprical Rank
      "MAP": 0.0, # Mean Average Precision
      **{f"Hits@{k}": 0.0 for k in ks},
      **{f"P@{k}": 0.0 for k in ks}, # Precision@k
      **{f"R@{k}": 0.0 for k in ks}, # Recall@k
      **{f"F1@{k}": 0.0 for k in ks}, # F1@k
      **{f"nDCG@{k}": 0.0 for k in ks}, # normalised Discounted Cumlative Gain @ k
      "MR": 0.0, # Mean Rank
      "aRP": 0.0  # R-Precision
    }
    # Median Rank & Coverage are calculated during the test procedure
    hit_count = 0 # for coverage
    total_possible_hits = 0 # for coverage := hit_count / total_possible_hits .. essentially: recall@k, when k = MAX_K
    all_ranks = [] # for median rank

    if model_name[0:3] == "HiT" and model_name[12:14] == "14" and model_name[15:16] == "F":
      centripetal_weight = 0.8
      print(f"Detected HiT -> 2014 <- ON SNOMED --> FULL <-- ... Setting centripetal weight to {centripetal_weight} ... ")
    elif model_name[0:3] == "HiT" and model_name[12:14] == "14" and model_name[15:16] == "M":
      centripetal_weight = 0.7
      print(f"Detected HiT -> 2014 <- ON SNOMED --> MINI <-- ... Setting centripetal weight to {centripetal_weight} ... ")
    elif model_name[0:3] == "OnT" and model_name[12:14] == "14" and model_name[15:16] == "F":
      centripetal_weight = 0.5
      print(f"Detected OnT -> 2014 <- ON SNOMED --> FULL <-- ... Setting centripetal weight to {centripetal_weight} ... ")
    elif model_name[0:3] == "OnT" and model_name[12:14] == "14" and model_name[15:16] == "M":
      centripetal_weight = 0.4
      print(f"Detected OnT -> 2014 <- ON SNOMED --> MINI <-- ... Setting centripetal weight to {centripetal_weight} ... ")
    elif model_name[0:3] == "HiT" and model_name[12:14] == "25" and model_name[15:16] == "F":
      centripetal_weight = 0.4
      print(f"Detected HiT -> 2025 <- on SNOMED --> FULL <-- ... Setting centripetal weight to {centripetal_weight} ... ")
    elif model_name[0:3] == "HiT" and model_name[12:14] == "25" and model_name[15:16] == "M":
      centripetal_weight = 0.8
      print(f"Detected HiT -> 2025 <- on SNOMED --> MINI <-- ... Setting centripetal weight to {centripetal_weight} ... ")
    elif model_name[0:3] == "OnT" and model_name[12:14] == "25" and model_name[15:16] == "F":
      centripetal_weight = 0.6
      print(f"Detected OnT -> 2025 <- on SNOMED --> FULL <-- ... Setting centripetal weight to {centripetal_weight} ... ")
    elif model_name[0:3] == "OnT" and model_name[12:14] == "25" and model_name[15:16] == "M":
      centripetal_weight = 0.6
      print(f"Detected OnT -> 2025 <- on SNOMED --> MINI <-- ... Setting centripetal weight to {centripetal_weight} ... ")
    else:
      print("Model centri weight not specified, using default (0.7)")

    print("#####################################")
    print(f"VERIFY CENTRI WEIGHT, VALUE SET TO: {centripetal_weight}")
    print("#####################################")

    for q_idx, query in enumerate(subsumpt_queries):
        
        qstr = query.get_query_string()
        gold_targets = query.get_unique_sorted_subsumptive_targets(key="depth", reverse=False, depth_cutoff=global_cutoff_depth) # [*parents, *ancestors]
        g_target_iris = set([x["iri"] for x in gold_targets])
        num_targets = len(g_target_iris)
        total_possible_hits += num_targets
        average_precision = 0.0
        hit_count_this_query = 0
        hit_count_lt_or_eq_num_targets = 0

        ranked_results: list[QueryResult] = [] # empty lists should not exist, but are treated as full misses
        
        #
        if isinstance(model, HiTRetriever):
          if model._score_fn == entity_subsumption:
            ranked_results = model.retrieve(qstr, top_k=MAX_K, reverse_candidate_scores=True, model=model._model, weight=centripetal_weight)
          elif model._score_fn == batch_poincare_dist_with_adaptive_curv_k: 
            ranked_results = model.retrieve(qstr, top_k=MAX_K, reverse_candidate_scores=False, model=model._model)
        #
        elif isinstance(model, OnTRetriever):
          if model._score_fn == concept_subsumption:
            ranked_results = model.retrieve(qstr, top_k=MAX_K, reverse_candidate_scores=True, model=model._model, weight=centripetal_weight)
          elif model._score_fn == batch_poincare_dist_with_adaptive_curv_k: 
            ranked_results = model.retrieve(qstr, top_k=MAX_K, reverse_candidate_scores=False, model=model._model)
        #
        elif isinstance(model, SBERTRetriever):
          if model._score_fn == batch_cosine_similarity:
            ranked_results = model.retrieve(qstr, top_k=MAX_K, reverse_candidate_scores=True)
          else:
            ranked_results = model.retrieve(qstr, top_k=MAX_K, reverse_candidate_scores=False)
        #
        elif isinstance(model, SapBERTRetriever):
          ranked_results = model.retrieve(qstr, top_k=MAX_K, reverse_candidate_scores=True)
        #
        elif isinstance(model, TFIDFRetriever):
          ranked_results = model.retrieve(qstr, top_k=MAX_K)
        #
        elif isinstance(model, BM25Retriever):
          ranked_results = model.retrieve(qstr, top_k=MAX_K)
        else:
           raise ValueError("No appropriate retriever has been set.")

        retrieved_iris = [iri for (_, iri, _, _) in ranked_results] # type: ignore

        # MRR & Mean Rank (on the first hit)
        rank_pos = None
        for rank_idx, iri in enumerate(retrieved_iris, start=1):
            if iri in g_target_iris:
                rank_pos = rank_idx
                results["MRR"] += 1.0 / rank_idx
                results["MR"] += rank_idx
                break
        
        # Average Precision (this query), for use in calculating mAP
        for rank_idx, iri in enumerate(retrieved_iris, start=1):
           if iri in g_target_iris:
              hit_count += 1
              hit_count_this_query += 1
              average_precision += hit_count_this_query / rank_idx
        average_precision /= num_targets
        results["MAP"] += average_precision

        # R-Precision (this query)
        for rank_idx, iri in enumerate(retrieved_iris, start=1):
           if iri in g_target_iris:
              hit_count_lt_or_eq_num_targets += 1
           if rank_idx == num_targets: # then we need to calculate the precision @ this index
              results["aRP"] += hit_count_lt_or_eq_num_targets / num_targets
              break

        # include a penalty to appropriately offset the MR
        # rather than artifically inflating the performance
        # by simply dropping queries that do not contain 
        # (unlikely in this case)
        if rank_pos is None:
            print("WARNING: IF MAX_K IS SET TO THE DOCUMENT SET SIZE, THERE IS A NON-EXISTING TARGET IN THE EDGE SET!")
            results["MR"] += MAX_K + 1 # penalty: rank := MAX_K + 1

        for k in ks:
            hit = 1 if (rank_pos is not None and rank_pos <= k) else 0
            results[f"Hits@{k}"] += hit
            top_k_results = set(retrieved_iris[:k])
            total_hits_at_k = len(g_target_iris.intersection(top_k_results))
            p_at_k = total_hits_at_k / k # Precision@K
            results[f"P@{k}"] += p_at_k
            r_at_k = total_hits_at_k / num_targets
            results[f"R@{k}"] += r_at_k
            if (p_at_k + r_at_k) > 0:
               results[f"F1@{k}"] += 2 * (p_at_k * r_at_k) / (p_at_k + r_at_k)
            iDCG, targets_with_dcg = query.get_targets_with_dcg(type="exp", depth_cutoff=global_cutoff_depth)
            results[f"nDCG@{k}"] += compute_ndcg_at_k(ranked_results, targets_with_dcg, k) # type: ignore

        final_rank = rank_pos if rank_pos is not None else MAX_K + 1
        all_ranks.append(final_rank)

    # normalise over queries & compute coverage
    N = len(subsumpt_queries)
    normalized = {metric: value / N for metric, value in results.items()}
    normalized['Cov'] = (hit_count / total_possible_hits) # calculate the coverage of this model
    normalized['Med'] = statistics.median(all_ranks) # median rank
    # area under precision-recall curve (trapezodial rule)
    recall_at_k_xs    = [normalized[f"R@{k}"] for k in ks]

    print(f"Model: {model_name}")
    print(f" mAP: \t  {normalized['MAP']:.2f}") # Mean Average Precision
    print(f" MRR*:   {normalized['MRR']:.2f}") # MRR at first hit ranks
    for k in [1, 3, 5]:
        print(f"  Hits@{k}:    {normalized[f'Hits@{k}']:.2f}")
    print(f" Med:     {normalized['Med']:.2f}") # Median Rank
    print(f" MR:      {normalized['MR']:.2f} ") # Mean Rank
    print(f" nDCG@10: {normalized['nDCG@10']:.2f}") # nDCG@10
    print(f" R@100:   {normalized['R@100']:.2f}") # Recall@100
    print("-"*60)

    all_results[model_name] = normalized

    model_metric_string = model_name.split()

    experiment_table.add_row([
      model_metric_string[0],
      model_metric_string[1],
      model_metric_string[2],
      normalized['MAP'], 
      normalized['MRR'],
      normalized['Hits@1'], 
      normalized['Hits@3'], 
      normalized['Hits@5'],
      normalized['Med'], 
      normalized['MR'],
      normalized['nDCG@10'],
      normalized['R@100']
    ])

if logs_path:
   print(f"Detected 'logs_path' set to {logs_path}")
else:
   print(f"--> !! LOGS PATH NOT DETECTED. SAVING TO '../logs' !! <--")
   logs_path = '../logs'

Path(logs_path).mkdir(parents=True, exist_ok=True)
output_file = f'{logs_path}/oov_entity_mentions_d_equals_one_results_DISEASE_ALL_4_EPOCHS_MINI_{get_date_string()}.json'
with open(output_file, 'w') as f:
    json.dump(all_results, f, indent=2)

print(f"All results saved to {output_file}")

print(f"Printing table: \n\n")

print(experiment_table.draw())

print("\n\n Printing LaTeX: \n\n")

print(latextable.draw_latex(
    table=experiment_table,
    caption="Performance of fetching single targets using OOV mentions with lambda set to eval values (Previous Eval Dataset)",
    use_booktabs=True, position="H", caption_above=True, caption_short="Single target performance of OOV mentions",
    label="tab:single-target-oov-results"
  )
)

Model centri weight not specified, using default (0.7)
#####################################
VERIFY CENTRI WEIGHT, VALUE SET TO: 0.7
#####################################
Model: BoW Lexical TFIDF
 mAP: 	  0.08
 MRR*:   0.08
  Hits@1:    0.05
  Hits@3:    0.10
  Hits@5:    0.11
 Med:     173179.00
 MR:      95399.54 
 nDCG@10: 0.09
 R@100:   0.17
------------------------------------------------------------
Model centri weight not specified, using default (0.7)
#####################################
VERIFY CENTRI WEIGHT, VALUE SET TO: 0.7
#####################################
Model: BoW Lexical BM25
 mAP: 	  0.08
 MRR*:   0.08
  Hits@1:    0.06
  Hits@3:    0.07
  Hits@5:    0.09
 Med:     17133.50
 MR:      40447.30 
 nDCG@10: 0.08
 R@100:   0.25
------------------------------------------------------------
Model centri weight not specified, using default (0.7)
#####################################
VERIFY CENTRI WEIGHT, VALUE SET TO: 0.7
#####################################
Model: SBERT 

/tmp/ipykernel_396855/2701625593.py:6: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  parent_emb_t = torch.Tensor(parent_emd)


Model: HiT-MxR SNO-14-M s_e
 mAP: 	  0.14
 MRR*:   0.14
  Hits@1:    0.10
  Hits@3:    0.16
  Hits@5:    0.16
 Med:     248.50
 MR:      11742.67 
 nDCG@10: 0.15
 R@100:   0.45
------------------------------------------------------------
Detected HiT -> 2014 <- ON SNOMED --> FULL <-- ... Setting centripetal weight to 0.8 ... 
#####################################
VERIFY CENTRI WEIGHT, VALUE SET TO: 0.8
#####################################
Model: HiT-MxR SNO-14-F d_k
 mAP: 	  0.19
 MRR*:   0.19
  Hits@1:    0.11
  Hits@3:    0.20
  Hits@5:    0.28
 Med:     69.00
 MR:      11837.81 
 nDCG@10: 0.22
 R@100:   0.54
------------------------------------------------------------
Detected HiT -> 2014 <- ON SNOMED --> FULL <-- ... Setting centripetal weight to 0.8 ... 
#####################################
VERIFY CENTRI WEIGHT, VALUE SET TO: 0.8
#####################################
Model: HiT-MxR SNO-14-F s_e
 mAP: 	  0.08
 MRR*:   0.08
  Hits@1:    0.05
  Hits@3:    0.09
  Hits@5:    0.10
 Me

In [46]:
global_cutoff_depth = 3

In [47]:
# PREP TABLE START #
experiment_table = texttable.Texttable()
experiment_table.set_deco(texttable.Texttable.HEADER)
experiment_table.set_precision(2)
experiment_table.set_cols_dtype(['t', 't', 't', 'f', 'f', 'f', 'f', 'f', 'f', 'f', 'f', 'f'])
experiment_table.set_cols_align(["l", "l", "l", "c", "c", "c", "c", "c", "c", "c", "c", "c"])
experiment_table.header(["Model", "Variant", "Metric", "mAP", "MRR", "H@1", "H@3", "H@5", "Med", "MR", "nDCG@10", "R@100"])
# END-PREP TABLE #

ks      = [1, 3, 5, 10, 100, len(entity_verbalisation_list)]
MAX_K   = max(ks)

all_results = {}

centripetal_weight = 0.7

for model_name, model in models_dict.items():
    
    # init accumulators
    results = {
      "MRR": 0.0, # Mean Reciprical Rank
      "MAP": 0.0, # Mean Average Precision
      **{f"Hits@{k}": 0.0 for k in ks},
      **{f"P@{k}": 0.0 for k in ks}, # Precision@k
      **{f"R@{k}": 0.0 for k in ks}, # Recall@k
      **{f"F1@{k}": 0.0 for k in ks}, # F1@k
      **{f"nDCG@{k}": 0.0 for k in ks}, # normalised Discounted Cumlative Gain @ k
      "MR": 0.0, # Mean Rank
      "aRP": 0.0  # R-Precision
    }
    # Median Rank & Coverage are calculated during the test procedure
    hit_count = 0 # for coverage
    total_possible_hits = 0 # for coverage := hit_count / total_possible_hits .. essentially: recall@k, when k = MAX_K
    all_ranks = [] # for median rank

    if model_name[0:3] == "HiT" and model_name[12:14] == "14" and model_name[15:16] == "F":
      centripetal_weight = 0.8
      print(f"Detected HiT -> 2014 <- ON SNOMED --> FULL <-- ... Setting centripetal weight to {centripetal_weight} ... ")
    elif model_name[0:3] == "HiT" and model_name[12:14] == "14" and model_name[15:16] == "M":
      centripetal_weight = 0.7
      print(f"Detected HiT -> 2014 <- ON SNOMED --> MINI <-- ... Setting centripetal weight to {centripetal_weight} ... ")
    elif model_name[0:3] == "OnT" and model_name[12:14] == "14" and model_name[15:16] == "F":
      centripetal_weight = 0.5
      print(f"Detected OnT -> 2014 <- ON SNOMED --> FULL <-- ... Setting centripetal weight to {centripetal_weight} ... ")
    elif model_name[0:3] == "OnT" and model_name[12:14] == "14" and model_name[15:16] == "M":
      centripetal_weight = 0.4
      print(f"Detected OnT -> 2014 <- ON SNOMED --> MINI <-- ... Setting centripetal weight to {centripetal_weight} ... ")
    elif model_name[0:3] == "HiT" and model_name[12:14] == "25" and model_name[15:16] == "F":
      centripetal_weight = 0.4
      print(f"Detected HiT -> 2025 <- on SNOMED --> FULL <-- ... Setting centripetal weight to {centripetal_weight} ... ")
    elif model_name[0:3] == "HiT" and model_name[12:14] == "25" and model_name[15:16] == "M":
      centripetal_weight = 0.8
      print(f"Detected HiT -> 2025 <- on SNOMED --> MINI <-- ... Setting centripetal weight to {centripetal_weight} ... ")
    elif model_name[0:3] == "OnT" and model_name[12:14] == "25" and model_name[15:16] == "F":
      centripetal_weight = 0.6
      print(f"Detected OnT -> 2025 <- on SNOMED --> FULL <-- ... Setting centripetal weight to {centripetal_weight} ... ")
    elif model_name[0:3] == "OnT" and model_name[12:14] == "25" and model_name[15:16] == "M":
      centripetal_weight = 0.6
      print(f"Detected OnT -> 2025 <- on SNOMED --> MINI <-- ... Setting centripetal weight to {centripetal_weight} ... ")
    else:
      print("Model centri weight not specified, using default (0.7)")

    print("#####################################")
    print(f"VERIFY CENTRI WEIGHT, VALUE SET TO: {centripetal_weight}")
    print("#####################################")

    for q_idx, query in enumerate(subsumpt_queries):
        
        qstr = query.get_query_string()
        gold_targets = query.get_unique_sorted_subsumptive_targets(key="depth", reverse=False, depth_cutoff=global_cutoff_depth) # [*parents, *ancestors]
        g_target_iris = set([x["iri"] for x in gold_targets])
        num_targets = len(g_target_iris)
        total_possible_hits += num_targets
        average_precision = 0.0
        hit_count_this_query = 0
        hit_count_lt_or_eq_num_targets = 0

        ranked_results: list[QueryResult] = [] # empty lists should not exist, but are treated as full misses
        
        #
        if isinstance(model, HiTRetriever):
          if model._score_fn == entity_subsumption:
            ranked_results = model.retrieve(qstr, top_k=MAX_K, reverse_candidate_scores=True, model=model._model, weight=centripetal_weight)
          elif model._score_fn == batch_poincare_dist_with_adaptive_curv_k: 
            ranked_results = model.retrieve(qstr, top_k=MAX_K, reverse_candidate_scores=False, model=model._model)
        #
        elif isinstance(model, OnTRetriever):
          if model._score_fn == concept_subsumption:
            ranked_results = model.retrieve(qstr, top_k=MAX_K, reverse_candidate_scores=True, model=model._model, weight=centripetal_weight)
          elif model._score_fn == batch_poincare_dist_with_adaptive_curv_k: 
            ranked_results = model.retrieve(qstr, top_k=MAX_K, reverse_candidate_scores=False, model=model._model)
        #
        elif isinstance(model, SBERTRetriever):
          if model._score_fn == batch_cosine_similarity:
            ranked_results = model.retrieve(qstr, top_k=MAX_K, reverse_candidate_scores=True)
          else:
            ranked_results = model.retrieve(qstr, top_k=MAX_K, reverse_candidate_scores=False)
        #
        elif isinstance(model, SapBERTRetriever):
          ranked_results = model.retrieve(qstr, top_k=MAX_K, reverse_candidate_scores=True)
        #
        elif isinstance(model, TFIDFRetriever):
          ranked_results = model.retrieve(qstr, top_k=MAX_K)
        #
        elif isinstance(model, BM25Retriever):
          ranked_results = model.retrieve(qstr, top_k=MAX_K)
        else:
           raise ValueError("No appropriate retriever has been set.")

        retrieved_iris = [iri for (_, iri, _, _) in ranked_results] # type: ignore

        # MRR & Mean Rank (on the first hit)
        rank_pos = None
        for rank_idx, iri in enumerate(retrieved_iris, start=1):
            if iri in g_target_iris:
                rank_pos = rank_idx
                results["MRR"] += 1.0 / rank_idx
                results["MR"] += rank_idx
                break
        
        # Average Precision (this query), for use in calculating mAP
        for rank_idx, iri in enumerate(retrieved_iris, start=1):
           if iri in g_target_iris:
              hit_count += 1
              hit_count_this_query += 1
              average_precision += hit_count_this_query / rank_idx
        average_precision /= num_targets
        results["MAP"] += average_precision

        # R-Precision (this query)
        for rank_idx, iri in enumerate(retrieved_iris, start=1):
           if iri in g_target_iris:
              hit_count_lt_or_eq_num_targets += 1
           if rank_idx == num_targets: # then we need to calculate the precision @ this index
              results["aRP"] += hit_count_lt_or_eq_num_targets / num_targets
              break

        # include a penalty to appropriately offset the MR
        # rather than artifically inflating the performance
        # by simply dropping queries that do not contain 
        # (unlikely in this case)
        if rank_pos is None:
            print("WARNING: IF MAX_K IS SET TO THE DOCUMENT SET SIZE, THERE IS A NON-EXISTING TARGET IN THE EDGE SET!")
            results["MR"] += MAX_K + 1 # penalty: rank := MAX_K + 1

        for k in ks:
            hit = 1 if (rank_pos is not None and rank_pos <= k) else 0
            results[f"Hits@{k}"] += hit
            top_k_results = set(retrieved_iris[:k])
            total_hits_at_k = len(g_target_iris.intersection(top_k_results))
            p_at_k = total_hits_at_k / k # Precision@K
            results[f"P@{k}"] += p_at_k
            r_at_k = total_hits_at_k / num_targets
            results[f"R@{k}"] += r_at_k
            if (p_at_k + r_at_k) > 0:
               results[f"F1@{k}"] += 2 * (p_at_k * r_at_k) / (p_at_k + r_at_k)
            iDCG, targets_with_dcg = query.get_targets_with_dcg(type="exp", depth_cutoff=global_cutoff_depth)
            results[f"nDCG@{k}"] += compute_ndcg_at_k(ranked_results, targets_with_dcg, k) # type: ignore

        final_rank = rank_pos if rank_pos is not None else MAX_K + 1
        all_ranks.append(final_rank)

    # normalise over queries & compute coverage
    N = len(subsumpt_queries)
    normalized = {metric: value / N for metric, value in results.items()}
    normalized['Cov'] = (hit_count / total_possible_hits) # calculate the coverage of this model
    normalized['Med'] = statistics.median(all_ranks) # median rank
    # area under precision-recall curve (trapezodial rule)
    recall_at_k_xs    = [normalized[f"R@{k}"] for k in ks]

    print(f"Model: {model_name}")
    print(f" mAP: \t  {normalized['MAP']:.2f}") # Mean Average Precision
    print(f" MRR*:   {normalized['MRR']:.2f}") # MRR at first hit ranks
    for k in [1, 3, 5]:
        print(f"  Hits@{k}:    {normalized[f'Hits@{k}']:.2f}")
    print(f" Med:     {normalized['Med']:.2f}") # Median Rank
    print(f" MR:      {normalized['MR']:.2f} ") # Mean Rank
    print(f" nDCG@10: {normalized['nDCG@10']:.2f}") # nDCG@10
    print(f" R@100:   {normalized['R@100']:.2f}") # Recall@100
    print("-"*60)

    all_results[model_name] = normalized

    model_metric_string = model_name.split()

    experiment_table.add_row([
      model_metric_string[0],
      model_metric_string[1],
      model_metric_string[2],
      normalized['MAP'], 
      normalized['MRR'],
      normalized['Hits@1'], 
      normalized['Hits@3'], 
      normalized['Hits@5'],
      normalized['Med'], 
      normalized['MR'],
      normalized['nDCG@10'],
      normalized['R@100']
    ])

if logs_path:
   print(f"Detected 'logs_path' set to {logs_path}")
else:
   print(f"--> !! LOGS PATH NOT DETECTED. SAVING TO '../logs' !! <--")
   logs_path = '../logs'

Path(logs_path).mkdir(parents=True, exist_ok=True)
output_file = f'{logs_path}/oov_entity_mentions_d_equals_three_results_DISEASE_ALL_4_EPOCHS_MINI_{get_date_string()}.json'
with open(output_file, 'w') as f:
    json.dump(all_results, f, indent=2)

print(f"All results saved to {output_file}")

print(f"Printing table: \n\n")

print(experiment_table.draw())

print("\n\n Printing LaTeX: \n\n")

print(latextable.draw_latex(
    table=experiment_table,
    caption="Performance of fetching multiple relevant targets (d=3) using OOV mentions with lambda set to eval values (Previous Eval Dataset)",
    use_booktabs=True, position="H", caption_above=True, caption_short="Multiple target performance d=3 of OOV mentions",
    label="tab:multi-target-oov-weighted-3"
  )
)

Model centri weight not specified, using default (0.7)
#####################################
VERIFY CENTRI WEIGHT, VALUE SET TO: 0.7
#####################################
Model: BoW Lexical TFIDF
 mAP: 	  0.03
 MRR*:   0.09
  Hits@1:    0.05
  Hits@3:    0.11
  Hits@5:    0.12
 Med:     173179.00
 MR:      91027.24 
 nDCG@10: 0.07
 R@100:   0.08
------------------------------------------------------------
Model centri weight not specified, using default (0.7)
#####################################
VERIFY CENTRI WEIGHT, VALUE SET TO: 0.7
#####################################
Model: BoW Lexical BM25
 mAP: 	  0.04
 MRR*:   0.09
  Hits@1:    0.07
  Hits@3:    0.09
  Hits@5:    0.10
 Med:     17133.50
 MR:      31953.41 
 nDCG@10: 0.07
 R@100:   0.12
------------------------------------------------------------
Model centri weight not specified, using default (0.7)
#####################################
VERIFY CENTRI WEIGHT, VALUE SET TO: 0.7
#####################################
Model: SBERT 

In [48]:
global_cutoff_depth = 5

In [49]:
# PREP TABLE START #
experiment_table = texttable.Texttable()
experiment_table.set_deco(texttable.Texttable.HEADER)
experiment_table.set_precision(2)
experiment_table.set_cols_dtype(['t', 't', 't', 'f', 'f', 'f', 'f', 'f', 'f', 'f', 'f', 'f'])
experiment_table.set_cols_align(["l", "l", "l", "c", "c", "c", "c", "c", "c", "c", "c", "c"])
experiment_table.header(["Model", "Variant", "Metric", "mAP", "MRR", "H@1", "H@3", "H@5", "Med", "MR", "nDCG@10", "R@100"])
# END-PREP TABLE #

ks      = [1, 3, 5, 10, 100, len(entity_verbalisation_list)]
MAX_K   = max(ks)

all_results = {}

centripetal_weight = 0.7

for model_name, model in models_dict.items():
    
    # init accumulators
    results = {
      "MRR": 0.0, # Mean Reciprical Rank
      "MAP": 0.0, # Mean Average Precision
      **{f"Hits@{k}": 0.0 for k in ks},
      **{f"P@{k}": 0.0 for k in ks}, # Precision@k
      **{f"R@{k}": 0.0 for k in ks}, # Recall@k
      **{f"F1@{k}": 0.0 for k in ks}, # F1@k
      **{f"nDCG@{k}": 0.0 for k in ks}, # normalised Discounted Cumlative Gain @ k
      "MR": 0.0, # Mean Rank
      "aRP": 0.0  # R-Precision
    }
    # Median Rank & Coverage are calculated during the test procedure
    hit_count = 0 # for coverage
    total_possible_hits = 0 # for coverage := hit_count / total_possible_hits .. essentially: recall@k, when k = MAX_K
    all_ranks = [] # for median rank

    if model_name[0:3] == "HiT" and model_name[12:14] == "14" and model_name[15:16] == "F":
      centripetal_weight = 0.8
      print(f"Detected HiT -> 2014 <- ON SNOMED --> FULL <-- ... Setting centripetal weight to {centripetal_weight} ... ")
    elif model_name[0:3] == "HiT" and model_name[12:14] == "14" and model_name[15:16] == "M":
      centripetal_weight = 0.7
      print(f"Detected HiT -> 2014 <- ON SNOMED --> MINI <-- ... Setting centripetal weight to {centripetal_weight} ... ")
    elif model_name[0:3] == "OnT" and model_name[12:14] == "14" and model_name[15:16] == "F":
      centripetal_weight = 0.5
      print(f"Detected OnT -> 2014 <- ON SNOMED --> FULL <-- ... Setting centripetal weight to {centripetal_weight} ... ")
    elif model_name[0:3] == "OnT" and model_name[12:14] == "14" and model_name[15:16] == "M":
      centripetal_weight = 0.4
      print(f"Detected OnT -> 2014 <- ON SNOMED --> MINI <-- ... Setting centripetal weight to {centripetal_weight} ... ")
    elif model_name[0:3] == "HiT" and model_name[12:14] == "25" and model_name[15:16] == "F":
      centripetal_weight = 0.4
      print(f"Detected HiT -> 2025 <- on SNOMED --> FULL <-- ... Setting centripetal weight to {centripetal_weight} ... ")
    elif model_name[0:3] == "HiT" and model_name[12:14] == "25" and model_name[15:16] == "M":
      centripetal_weight = 0.8
      print(f"Detected HiT -> 2025 <- on SNOMED --> MINI <-- ... Setting centripetal weight to {centripetal_weight} ... ")
    elif model_name[0:3] == "OnT" and model_name[12:14] == "25" and model_name[15:16] == "F":
      centripetal_weight = 0.6
      print(f"Detected OnT -> 2025 <- on SNOMED --> FULL <-- ... Setting centripetal weight to {centripetal_weight} ... ")
    elif model_name[0:3] == "OnT" and model_name[12:14] == "25" and model_name[15:16] == "M":
      centripetal_weight = 0.6
      print(f"Detected OnT -> 2025 <- on SNOMED --> MINI <-- ... Setting centripetal weight to {centripetal_weight} ... ")
    else:
      print("Model centri weight not specified, using default (0.7)")

    print("#####################################")
    print(f"VERIFY CENTRI WEIGHT, VALUE SET TO: {centripetal_weight}")
    print("#####################################")

    for q_idx, query in enumerate(subsumpt_queries):
        
        qstr = query.get_query_string()
        gold_targets = query.get_unique_sorted_subsumptive_targets(key="depth", reverse=False, depth_cutoff=global_cutoff_depth) # [*parents, *ancestors]
        g_target_iris = set([x["iri"] for x in gold_targets])
        num_targets = len(g_target_iris)
        total_possible_hits += num_targets
        average_precision = 0.0
        hit_count_this_query = 0
        hit_count_lt_or_eq_num_targets = 0

        ranked_results: list[QueryResult] = [] # empty lists should not exist, but are treated as full misses
        
        #
        if isinstance(model, HiTRetriever):
          if model._score_fn == entity_subsumption:
            ranked_results = model.retrieve(qstr, top_k=MAX_K, reverse_candidate_scores=True, model=model._model, weight=centripetal_weight)
          elif model._score_fn == batch_poincare_dist_with_adaptive_curv_k: 
            ranked_results = model.retrieve(qstr, top_k=MAX_K, reverse_candidate_scores=False, model=model._model)
        #
        elif isinstance(model, OnTRetriever):
          if model._score_fn == concept_subsumption:
            ranked_results = model.retrieve(qstr, top_k=MAX_K, reverse_candidate_scores=True, model=model._model, weight=centripetal_weight)
          elif model._score_fn == batch_poincare_dist_with_adaptive_curv_k: 
            ranked_results = model.retrieve(qstr, top_k=MAX_K, reverse_candidate_scores=False, model=model._model)
        #
        elif isinstance(model, SBERTRetriever):
          if model._score_fn == batch_cosine_similarity:
            ranked_results = model.retrieve(qstr, top_k=MAX_K, reverse_candidate_scores=True)
          else:
            ranked_results = model.retrieve(qstr, top_k=MAX_K, reverse_candidate_scores=False)
        #
        elif isinstance(model, SapBERTRetriever):
          ranked_results = model.retrieve(qstr, top_k=MAX_K, reverse_candidate_scores=True)
        #
        elif isinstance(model, TFIDFRetriever):
          ranked_results = model.retrieve(qstr, top_k=MAX_K)
        #
        elif isinstance(model, BM25Retriever):
          ranked_results = model.retrieve(qstr, top_k=MAX_K)
        else:
           raise ValueError("No appropriate retriever has been set.")

        retrieved_iris = [iri for (_, iri, _, _) in ranked_results] # type: ignore

        # MRR & Mean Rank (on the first hit)
        rank_pos = None
        for rank_idx, iri in enumerate(retrieved_iris, start=1):
            if iri in g_target_iris:
                rank_pos = rank_idx
                results["MRR"] += 1.0 / rank_idx
                results["MR"] += rank_idx
                break
        
        # Average Precision (this query), for use in calculating mAP
        for rank_idx, iri in enumerate(retrieved_iris, start=1):
           if iri in g_target_iris:
              hit_count += 1
              hit_count_this_query += 1
              average_precision += hit_count_this_query / rank_idx
        average_precision /= num_targets
        results["MAP"] += average_precision

        # R-Precision (this query)
        for rank_idx, iri in enumerate(retrieved_iris, start=1):
           if iri in g_target_iris:
              hit_count_lt_or_eq_num_targets += 1
           if rank_idx == num_targets: # then we need to calculate the precision @ this index
              results["aRP"] += hit_count_lt_or_eq_num_targets / num_targets
              break

        # include a penalty to appropriately offset the MR
        # rather than artifically inflating the performance
        # by simply dropping queries that do not contain 
        # (unlikely in this case)
        if rank_pos is None:
            print("WARNING: IF MAX_K IS SET TO THE DOCUMENT SET SIZE, THERE IS A NON-EXISTING TARGET IN THE EDGE SET!")
            results["MR"] += MAX_K + 1 # penalty: rank := MAX_K + 1

        for k in ks:
            hit = 1 if (rank_pos is not None and rank_pos <= k) else 0
            results[f"Hits@{k}"] += hit
            top_k_results = set(retrieved_iris[:k])
            total_hits_at_k = len(g_target_iris.intersection(top_k_results))
            p_at_k = total_hits_at_k / k # Precision@K
            results[f"P@{k}"] += p_at_k
            r_at_k = total_hits_at_k / num_targets
            results[f"R@{k}"] += r_at_k
            if (p_at_k + r_at_k) > 0:
               results[f"F1@{k}"] += 2 * (p_at_k * r_at_k) / (p_at_k + r_at_k)
            iDCG, targets_with_dcg = query.get_targets_with_dcg(type="exp", depth_cutoff=global_cutoff_depth)
            results[f"nDCG@{k}"] += compute_ndcg_at_k(ranked_results, targets_with_dcg, k) # type: ignore

        final_rank = rank_pos if rank_pos is not None else MAX_K + 1
        all_ranks.append(final_rank)

    # normalise over queries & compute coverage
    N = len(subsumpt_queries)
    normalized = {metric: value / N for metric, value in results.items()}
    normalized['Cov'] = (hit_count / total_possible_hits) # calculate the coverage of this model
    normalized['Med'] = statistics.median(all_ranks) # median rank
    # area under precision-recall curve (trapezodial rule)
    recall_at_k_xs    = [normalized[f"R@{k}"] for k in ks]

    print(f"Model: {model_name}")
    print(f" mAP: \t  {normalized['MAP']:.2f}") # Mean Average Precision
    print(f" MRR*:   {normalized['MRR']:.2f}") # MRR at first hit ranks
    for k in [1, 3, 5]:
        print(f"  Hits@{k}:    {normalized[f'Hits@{k}']:.2f}")
    print(f" Med:     {normalized['Med']:.2f}") # Median Rank
    print(f" MR:      {normalized['MR']:.2f} ") # Mean Rank
    print(f" nDCG@10: {normalized['nDCG@10']:.2f}") # nDCG@10
    print(f" R@100:   {normalized['R@100']:.2f}") # Recall@100
    print("-"*60)

    all_results[model_name] = normalized

    model_metric_string = model_name.split()

    experiment_table.add_row([
      model_metric_string[0],
      model_metric_string[1],
      model_metric_string[2],
      normalized['MAP'], 
      normalized['MRR'],
      normalized['Hits@1'], 
      normalized['Hits@3'], 
      normalized['Hits@5'],
      normalized['Med'], 
      normalized['MR'],
      normalized['nDCG@10'],
      normalized['R@100']
    ])

if logs_path:
   print(f"Detected 'logs_path' set to {logs_path}")
else:
   print(f"--> !! LOGS PATH NOT DETECTED. SAVING TO '../logs' !! <--")
   logs_path = '../logs'

Path(logs_path).mkdir(parents=True, exist_ok=True)
output_file = f'{logs_path}/oov_entity_mentions_d_equals_five_results_DISEASE_ALL_4_EPOCHS_MINI_{get_date_string()}.json'
with open(output_file, 'w') as f:
    json.dump(all_results, f, indent=2)

print(f"All results saved to {output_file}")

print(f"Printing table: \n\n")

print(experiment_table.draw())

print("\n\n Printing LaTeX: \n\n")

print(latextable.draw_latex(
    table=experiment_table,
    caption="Performance of fetching multiple relevant targets (d=5) using OOV mentions with lambda set to eval values (Previous Eval Dataset)",
    use_booktabs=True, position="H", caption_above=True, caption_short="Multiple target performance d=5 of OOV mentions",
    label="tab:multi-target-oov-weighted-5"
  )
)

Model centri weight not specified, using default (0.7)
#####################################
VERIFY CENTRI WEIGHT, VALUE SET TO: 0.7
#####################################
Model: BoW Lexical TFIDF
 mAP: 	  0.02
 MRR*:   0.09
  Hits@1:    0.05
  Hits@3:    0.11
  Hits@5:    0.12
 Med:     173179.00
 MR:      90999.45 
 nDCG@10: 0.06
 R@100:   0.05
------------------------------------------------------------
Model centri weight not specified, using default (0.7)
#####################################
VERIFY CENTRI WEIGHT, VALUE SET TO: 0.7
#####################################
Model: BoW Lexical BM25
 mAP: 	  0.02
 MRR*:   0.09
  Hits@1:    0.07
  Hits@3:    0.09
  Hits@5:    0.10
 Med:     12546.00
 MR:      30385.46 
 nDCG@10: 0.06
 R@100:   0.07
------------------------------------------------------------
Model centri weight not specified, using default (0.7)
#####################################
VERIFY CENTRI WEIGHT, VALUE SET TO: 0.7
#####################################


Model: SBERT PLM cos-sim
 mAP: 	  0.05
 MRR*:   0.20
  Hits@1:    0.12
  Hits@3:    0.26
  Hits@5:    0.28
 Med:     43.00
 MR:      4483.52 
 nDCG@10: 0.15
 R@100:   0.19
------------------------------------------------------------
Model centri weight not specified, using default (0.7)
#####################################
VERIFY CENTRI WEIGHT, VALUE SET TO: 0.7
#####################################
Model: SapBERT FT cos-sim
 mAP: 	  0.07
 MRR*:   0.28
  Hits@1:    0.19
  Hits@3:    0.29
  Hits@5:    0.41
 Med:     12.00
 MR:      4246.10 
 nDCG@10: 0.22
 R@100:   0.19
------------------------------------------------------------
Detected HiT -> 2014 <- ON SNOMED --> MINI <-- ... Setting centripetal weight to 0.7 ... 
#####################################
VERIFY CENTRI WEIGHT, VALUE SET TO: 0.7
#####################################
Model: HiT-MxR SNO-14-M d_k
 mAP: 	  0.09
 MRR*:   0.24
  Hits@1:    0.12
  Hits@3:    0.29
  Hits@5:    0.34
 Med:     9.50
 MR:      530.41 
 nDCG@10: 0.1

In [50]:
from grouped_evaluation_4e import run_grouped_evaluation, compare_evaluation_approaches

In [51]:
compare_evaluation_approaches(
    models_dict,
    subsumpt_queries,
    entity_verbalisation_list,
    depth_cutoff=1
)

EVALUATION APPROACH COMPARISON

Original query count:     80
Grouped query count:      55
Queries with >1 target:   16

Target count distribution:
  Min targets per query:  1
  Max targets per query:  3
  Mean targets per query: 1.45
  Median targets:         1

Example queries with multiple targets (poly-hierarchy):
  'lynch syndrome': 2 targets
  'wolman': 3 targets
  'frontal fibrosing alopecia': 2 targets
  'waardenburg syndrome type iv': 3 targets
  'triplenegative breast cancer': 3 targets


In [52]:
compare_evaluation_approaches(
    models_dict,
    subsumpt_queries,
    entity_verbalisation_list,
    depth_cutoff=3
)

EVALUATION APPROACH COMPARISON

Original query count:     80
Grouped query count:      55
Queries with >1 target:   55

Target count distribution:
  Min targets per query:  3
  Max targets per query:  9
  Mean targets per query: 4.15
  Median targets:         3

Example queries with multiple targets (poly-hierarchy):
  'inflammatory skin disease': 3 targets
  'chronic kidney disease': 3 targets
  'lynch syndrome': 5 targets
  'wolman': 7 targets
  'ventricular tachyarrhythmias': 3 targets


In [53]:
compare_evaluation_approaches(
    models_dict,
    subsumpt_queries,
    entity_verbalisation_list,
    depth_cutoff=5
)

EVALUATION APPROACH COMPARISON

Original query count:     80
Grouped query count:      55
Queries with >1 target:   55

Target count distribution:
  Min targets per query:  5
  Max targets per query:  15
  Mean targets per query: 6.38
  Median targets:         5

Example queries with multiple targets (poly-hierarchy):
  'inflammatory skin disease': 5 targets
  'chronic kidney disease': 5 targets
  'lynch syndrome': 7 targets
  'wolman': 9 targets
  'ventricular tachyarrhythmias': 5 targets


In [54]:
from grouped_evaluation_4e import compare_evaluation_approaches_with_target_depth

In [55]:
compare_evaluation_approaches_with_target_depth(
  models_dict,
  subsumpt_queries,
  entity_verbalisation_list,
  depth_cutoff=1
)

EVALUATION APPROACH COMPARISON (depth_cutoff=1)

Original query count:     80
Grouped query count:      55
Queries from >1 source:   16

Target statistics (depth <= 1):
  Total targets:          168
  Min targets per query:  2
  Mean targets per query: 3.05
  Median targets:         2.0
  Max targets per query:  8

Queries with >1 target:   55 (100.0%)

Target count distribution:
    2 targets:   34 queries ( 61.8%) ██████████████████████████████
    3 targets:    5 queries (  9.1%) ████
    4 targets:    7 queries ( 12.7%) ██████
    6 targets:    7 queries ( 12.7%) ██████
    7 targets:    1 queries (  1.8%) 
    8 targets:    1 queries (  1.8%) 

Depth distribution across all targets (n=168):
  depth 0:    80 targets ( 47.6%) ███████████████████████
  depth 1:    88 targets ( 52.4%) ██████████████████████████

Top 5 queries by target count:
  'waardenburg syndrome type iv': 8 targets, depths [0-1]
  'wolman': 7 targets, depths [0-1]
  'triplenegative breast cancer': 6 targets, depth

In [56]:
compare_evaluation_approaches_with_target_depth(
  models_dict,
  subsumpt_queries,
  entity_verbalisation_list,
  depth_cutoff=3
)

EVALUATION APPROACH COMPARISON (depth_cutoff=3)

Original query count:     80
Grouped query count:      55
Queries from >1 source:   16

Target statistics (depth <= 3):
  Total targets:          321
  Min targets per query:  4
  Mean targets per query: 5.84
  Median targets:         4.0
  Max targets per query:  19

Queries with >1 target:   55 (100.0%)

Target count distribution:
    4 targets:   34 queries ( 61.8%) ██████████████████████████████
    6 targets:    3 queries (  5.5%) ██
    7 targets:    4 queries (  7.3%) ███
    8 targets:    4 queries (  7.3%) ███
    9 targets:    2 queries (  3.6%) █
   10 targets:    7 queries ( 12.7%) ██████
   19 targets:    1 queries (  1.8%) 

Depth distribution across all targets (n=321):
  depth 0:    76 targets ( 23.7%) ███████████
  depth 1:    80 targets ( 24.9%) ████████████
  depth 2:    83 targets ( 25.9%) ████████████
  depth 3:    82 targets ( 25.5%) ████████████

Top 5 queries by target count:
  'waardenburg syndrome type iv': 19 t

In [57]:
compare_evaluation_approaches_with_target_depth(
  models_dict,
  subsumpt_queries,
  entity_verbalisation_list,
  depth_cutoff=5
)

EVALUATION APPROACH COMPARISON (depth_cutoff=5)

Original query count:     80
Grouped query count:      55
Queries from >1 source:   16

Target statistics (depth <= 5):
  Total targets:          440
  Min targets per query:  5
  Mean targets per query: 8.00
  Median targets:         6.0
  Max targets per query:  24

Queries with >1 target:   55 (100.0%)

Target count distribution:
    5 targets:    1 queries (  1.8%) 
    6 targets:   33 queries ( 60.0%) ██████████████████████████████
    8 targets:    4 queries (  7.3%) ███
    9 targets:    2 queries (  3.6%) █
   10 targets:    2 queries (  3.6%) █
   11 targets:    3 queries (  5.5%) ██
   12 targets:    8 queries ( 14.5%) ███████
   14 targets:    1 queries (  1.8%) 
   24 targets:    1 queries (  1.8%) 

Depth distribution across all targets (n=440):
  depth 0:    76 targets ( 17.3%) ████████
  depth 1:    76 targets ( 17.3%) ████████
  depth 2:    73 targets ( 16.6%) ████████
  depth 3:    72 targets ( 16.4%) ████████
  depth 4:

In [58]:
from grouped_evaluation_4e import (
    print_dataset_statistics_latex,
    print_dataset_statistics_latex_horizontal
)

In [59]:
print_dataset_statistics_latex(
    subsumpt_queries,
    dataset_name="OET-DISEASE Dataset",
    caption="Dataset statistics for OET-CPP",
    label="tab:oet-cpp-stats"
)

\begin{table}[H]
  \caption{Dataset statistics for OET-CPP}
  \label{tab:oet-cpp-stats}
  \centering
  \begin{tabular}{lr}
    \toprule
    \textbf{Statistic} & \textbf{Value} \\
    \midrule
    Number of Records & 80 \\
    Mean Hierarchy Depth & 7.06 \\
    Queries with $>1$ Target & 55 \\
    Grouped Query Count & 55 \\
    \midrule
    \multicolumn{2}{l}{\textbf{Targets at Depth Cutoff}} \\
    \midrule
    $d \leq 1$: Min / Mean / Max & 2 / 3.05 / 8 \\
    $d \leq 3$: Min / Mean / Max & 4 / 5.84 / 19 \\
    $d \leq 5$: Min / Mean / Max & 5 / 8.00 / 24 \\
    \bottomrule
  \end{tabular}
\end{table}


In [60]:
print_dataset_statistics_latex_horizontal(
    subsumpt_queries,
    dataset_name="OET-DISEASE Dataset",
    caption="Dataset statistics for OET-CPP",
    label="tab:oet-cpp-stats-horiz"
)

\begin{table}[H]
  \caption{Dataset statistics for OET-CPP}
  \label{tab:oet-cpp-stats-horiz}
  \centering
  \resizebox{\textwidth}{!}{%
  \begin{tabular}{lcccccccccc}
    \toprule
    & & & & & \multicolumn{2}{c}{$d \leq 1$} & \multicolumn{2}{c}{$d \leq 3$} & \multicolumn{2}{c}{$d \leq 5$} \\
    \cmidrule(lr){6-7} \cmidrule(lr){8-9} \cmidrule(lr){10-11}
    \textbf{Dataset} & \textbf{Records} & \textbf{Mean Depth} & \textbf{Multi-Target} & \textbf{Grouped} & \textbf{Mean} & \textbf{Max} & \textbf{Mean} & \textbf{Max} & \textbf{Mean} & \textbf{Max} \\
    \midrule
    OET-DISEASE Dataset & 80 & 7.06 & 55 & 55 & 3.05 & 8 & 5.84 & 19 & 8.00 & 24 \\
    \bottomrule
  \end{tabular}%
  }
\end{table}


In [61]:
run_grouped_evaluation(
    models_dict,
    subsumpt_queries,
    entity_verbalisation_list,
    depth_cutoff=1,
    logs_path=logs_path,
)

Grouping queries by mention string...
Grouped 80 queries into 55 unique mentions
Reduction: 25 duplicate mentions merged
------------------------------------------------------------

##################################################
Evaluating: BoW Lexical TFIDF
##################################################
Model centri weight not specified, using default (0.7)
Evaluating BoW Lexical TFIDF with centripetal weight: 0.7

Results for BoW Lexical TFIDF:
  mAP:      0.0976
  MRR:      0.1196
  Hits@1:   0.0727
  Hits@3:   0.1455
  Hits@5:   0.1636
  Median:   2139.0
  MR:       82027.35
  nDCG@10:  0.0000
  R@100:    0.2030
------------------------------------------------------------

##################################################
Evaluating: BoW Lexical BM25
##################################################
Model centri weight not specified, using default (0.7)
Evaluating BoW Lexical BM25 with centripetal weight: 0.7

Results for BoW Lexical BM25:
  mAP:      0.1055
  MRR:      

{'BoW Lexical TFIDF': {'MRR': 0.1196226713651616,
  'MAP': 0.09755192849631882,
  'Hits@1': 0.07272727272727272,
  'Hits@3': 0.14545454545454545,
  'Hits@5': 0.16363636363636364,
  'Hits@10': 0.18181818181818182,
  'Hits@100': 0.2545454545454545,
  'Hits@173178': 0.5272727272727272,
  'P@1': 0.07272727272727272,
  'P@3': 0.048484848484848485,
  'P@5': 0.03272727272727272,
  'P@10': 0.01818181818181818,
  'P@100': 0.002545454545454545,
  'P@173178': 3.779610889058972e-06,
  'R@1': 0.0606060606060606,
  'R@3': 0.11515151515151514,
  'R@5': 0.13333333333333333,
  'R@10': 0.1515151515151515,
  'R@100': 0.20303030303030303,
  'R@173178': 0.47878787878787876,
  'F1@1': 0.06363636363636363,
  'F1@3': 0.06606060606060606,
  'F1@5': 0.0512987012987013,
  'F1@10': 0.031998304725577464,
  'F1@100': 0.005015932775459156,
  'F1@173178': 7.559146604035464e-06,
  'nDCG@1': 0.0,
  'nDCG@3': 0.0,
  'nDCG@5': 0.0,
  'nDCG@10': 0.0,
  'nDCG@100': 0.0,
  'nDCG@173178': 0.0,
  'MR': 82027.34545454546,
  'a

In [62]:
run_grouped_evaluation(
    models_dict,
    subsumpt_queries,
    entity_verbalisation_list,
    depth_cutoff=3,
    logs_path=logs_path,
)

Grouping queries by mention string...
Grouped 80 queries into 55 unique mentions
Reduction: 25 duplicate mentions merged
------------------------------------------------------------

##################################################
Evaluating: BoW Lexical TFIDF
##################################################
Model centri weight not specified, using default (0.7)
Evaluating BoW Lexical TFIDF with centripetal weight: 0.7

Results for BoW Lexical TFIDF:
  mAP:      0.0370
  MRR:      0.1281
  Hits@1:   0.0727
  Hits@3:   0.1636
  Hits@5:   0.1818
  Median:   446.0
  MR:       75686.04
  nDCG@10:  0.0000
  R@100:    0.0964
------------------------------------------------------------

##################################################
Evaluating: BoW Lexical BM25
##################################################
Model centri weight not specified, using default (0.7)
Evaluating BoW Lexical BM25 with centripetal weight: 0.7

Results for BoW Lexical BM25:
  mAP:      0.0476
  MRR:      0

{'BoW Lexical TFIDF': {'MRR': 0.12805843727488178,
  'MAP': 0.03703696517639433,
  'Hits@1': 0.07272727272727272,
  'Hits@3': 0.16363636363636364,
  'Hits@5': 0.18181818181818182,
  'Hits@10': 0.21818181818181817,
  'Hits@100': 0.2909090909090909,
  'Hits@173178': 0.5636363636363636,
  'P@1': 0.07272727272727272,
  'P@3': 0.05454545454545454,
  'P@5': 0.03636363636363636,
  'P@10': 0.021818181818181816,
  'P@100': 0.003454545454545456,
  'P@173178': 6.0893730990394535e-06,
  'R@1': 0.0202020202020202,
  'R@3': 0.04565656565656566,
  'R@5': 0.05171717171717172,
  'R@10': 0.06080808080808082,
  'R@100': 0.0964141414141414,
  'R@173178': 0.2595454545454546,
  'F1@1': 0.030909090909090907,
  'F1@3': 0.048484848484848485,
  'F1@5': 0.041688311688311684,
  'F1@10': 0.031412710096920614,
  'F1@100': 0.006638643890719832,
  'F1@173178': 1.2178398222588293e-05,
  'nDCG@1': 0.0,
  'nDCG@3': 0.0,
  'nDCG@5': 0.0,
  'nDCG@10': 0.0,
  'nDCG@100': 0.0,
  'nDCG@173178': 0.0,
  'MR': 75686.03636363636

In [63]:
run_grouped_evaluation(
    models_dict,
    subsumpt_queries,
    entity_verbalisation_list,
    depth_cutoff=5,
    logs_path=logs_path,
)

Grouping queries by mention string...
Grouped 80 queries into 55 unique mentions
Reduction: 25 duplicate mentions merged
------------------------------------------------------------

##################################################
Evaluating: BoW Lexical TFIDF
##################################################
Model centri weight not specified, using default (0.7)
Evaluating BoW Lexical TFIDF with centripetal weight: 0.7

Results for BoW Lexical TFIDF:
  mAP:      0.0230
  MRR:      0.1281
  Hits@1:   0.0727
  Hits@3:   0.1636
  Hits@5:   0.1818
  Median:   409.0
  MR:       75651.44
  nDCG@10:  0.0000
  R@100:    0.0662
------------------------------------------------------------

##################################################
Evaluating: BoW Lexical BM25
##################################################
Model centri weight not specified, using default (0.7)
Evaluating BoW Lexical BM25 with centripetal weight: 0.7

Results for BoW Lexical BM25:
  mAP:      0.0292
  MRR:      0

{'BoW Lexical TFIDF': {'MRR': 0.12812697872859452,
  'MAP': 0.022998651499966567,
  'Hits@1': 0.07272727272727272,
  'Hits@3': 0.16363636363636364,
  'Hits@5': 0.18181818181818182,
  'Hits@10': 0.21818181818181817,
  'Hits@100': 0.2909090909090909,
  'Hits@173178': 0.5636363636363636,
  'P@1': 0.07272727272727272,
  'P@3': 0.05454545454545454,
  'P@5': 0.03636363636363636,
  'P@10': 0.021818181818181816,
  'P@100': 0.0038181818181818195,
  'P@173178': 6.7193082472159495e-06,
  'R@1': 0.012121212121212123,
  'R@3': 0.028225108225108222,
  'R@5': 0.031861471861471855,
  'R@10': 0.03731601731601731,
  'R@100': 0.06624163715072807,
  'R@173178': 0.18236914600550963,
  'F1@1': 0.020454545454545454,
  'F1@3': 0.036565656565656565,
  'F1@5': 0.03333333333333333,
  'F1@10': 0.026944741532976824,
  'F1@100': 0.007179462218433809,
  'F1@173178': 1.3438041798384292e-05,
  'nDCG@1': 0.0,
  'nDCG@3': 0.0,
  'nDCG@5': 0.0,
  'nDCG@10': 0.0,
  'nDCG@100': 0.0,
  'nDCG@173178': 0.0,
  'MR': 75651.4363